# Reproducing LeMessurier et al. (2019) from CRCNS ssc-10

**Paper.** LeMessurier AM, Laboy-Juarez KJ, McLain K, Chen S, Nguyen T, Feldman DE (2019).
Enrichment drives emergence of functional columns and improves sensory coding in the whisker map
in L2/3 of mouse S1. *eLife* 8:e46321.
[doi:10.7554/eLife.46321](https://doi.org/10.7554/eLife.46321). The claims file was written from
the version of record (PMC6697414), with its Methods, Table 1 and figure supplement legends.

**Data set.** CRCNS ssc-10, "2-photon calcium imaging in S1 of enriched and normally housed mice
during individual deflection of 9 whiskers" (LeMessurier & Feldman 2019,
[doi:10.6080/K0DB801M](https://doi.org/10.6080/K0DB801M)), read from the public bucket at
`https://crcnsarchive.s3-us-west-2.amazonaws.com/ssc-10/live/`. CRCNS data sets have no versions,
so the version is recorded as `unversioned`; this is the only copy the archive holds (data
descriptor version 0.6, Aug 15 2019; `data.tar.gz` last modified 2026-02-11 on the bucket). The
paper cites the data set by its DOI without a version. The file read is `data.tar.gz`, MD5
`80bcef52304d27c9e039e7027fe09ff4` in the data set's `checksums.md5`; the notebook computes the MD5
of the stream it reads and compares it (data check). The data set's `sample_code.zip` (MD5
`c3cd644bb9072bd4b63c96de5496a119`) was read for method details and is cited where used; it is not
run.

**What the data set holds.** For each of 36 imaging fields (folders `<mouse>/<date>/[f#]`), three
MATLAB files: raw and neuropil fluorescence per movie (`processed_data.mat`), dF/F, stimulus
tables, trial-aligned traces, stored responsiveness p-values and ROI positions relative to the
nine barrel centres (`reduced_data.mat`), and acquisition settings (`settings.mat`). The
juxtacellular recordings of Figure 1F and the barrel areas of Figure 2 S2 are not in it.

**How it is read.** The 4.7 GB tar file is streamed once, without writing to disk, and each field
is reduced to per-trial evoked responses as it arrives (helper `ssc10.py`). This takes about ten
minutes; the whole notebook about half an hour.

| Claim | Tested |
|---|---|
| table1-mice-and-fields | tested |
| l23-cell-counts, l4-cell-counts | tested |
| fig1f-gcamp-matches-spikes | blocked: the cell-attached recordings are not in the data set |
| fig2a (two claims), fig2b, fig2c (two claims), figs2-1 | tested |
| figs2-2-barrel-size-unchanged | tested, with radii from the normalised ROI positions |
| figs3-1 (two claims), discussion-l23-nh-in-column-tuning, fig3a, fig3c | tested |
| fig4a, fig4b, fig4c, figs4-1, figs4-2, fig4d (three claims), fig4ae | tested |
| fig5a, fig5b, fig5eh, figs5-1, fig6, figs5-2 | tested |
| fig7b, fig7c, fig7d | tested |

In [ ]:
import time
import warnings
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import ssc10

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()

In [ ]:
# Stream data.tar.gz once; each field is reduced as it arrives (about ten minutes).
F = list(ssc10.stream_fields(neuropil=True, verbose=False))
print(f"{len(F)} imaging fields read from CRCNS ssc-10 (unversioned) in {time.time() - started:.0f} s")

## Data check

In [ ]:
info = pd.DataFrame([dict(field=f["field"], mouse=f["mouse"], layer=f["layer"], group=f["group"],
                          rois=f["n_rois"], rois_raw=f["n_rois_raw"], movies=f["n_movies"],
                          movies_stim=f["n_movies_stim"], no_stim=len(f["movies_without_stimuli"]),
                          fs=round(f["fs"], 2), npSub=f["npSub"], permTestType=f["permTestType"],
                          missing_whiskers=",".join(f["missing_whiskers"]),
                          blank_trials=f["bl"].shape[1], label9=f["n_blank_label9"],
                          in_barrel=int((f["barrel"] != "").sum()), nan_rois=int(f["has_nan"].sum()))
                     for f in F])
print(info.to_string(index=False))
print("\nFields and mice per layer and group:")
print(info.groupby(["layer", "group"]).agg(fields=("field", "size"), mice=("mouse", "nunique"),
                                           rois=("rois", "sum"), rois_raw=("rois_raw", "sum")).to_string())
print("\nFields where processed_data.mat holds more ROIs than reduced_data.mat:", int((info.rois_raw > info.rois).sum()), "of", len(info))
print("Fields with dF/F movies that have no Stimuli entry:", info[info.no_stim > 0].field.tolist())
print("Frame rates (Hz):", sorted(info.fs.unique()), "; Methods: 30 Hz")
print("framesEvoked (0-based) per field:", sorted({(int(f["frames_evoked"][0]), int(f["frames_evoked"][-1])) for f in F}),
      "; baseline frames:", sorted({f["n_pre"] for f in F}))
print("MD5 of the streamed data.tar.gz:", ssc10.STREAM_MD5.get("data.tar.gz"), "; checksums.md5:", ssc10.DATA_MD5,
      "; equal:", ssc10.STREAM_MD5.get("data.tar.gz") == ssc10.DATA_MD5)
print("Every field's Label code (k = whisk[k], 9 = blank) agrees with its per-whisker frame lists:",
      all(f["label_code_ok"] for f in F))
print("Trials per whisker, min and max over fields:",
      min(f["ev"][w].shape[1] for f in F for w in ssc10.WHISKERS if w not in f["missing_whiskers"]),
      max(f["ev"][w].shape[1] for f in F for w in ssc10.WHISKERS))

The folders hold 22 Drd3-Cre (L2/3) and 14 Scnn1a-Cre (L4) fields, every movie at 29.92 Hz
(Methods: 30 Hz), with an evoked window of 30 frames after 15 baseline frames, as the descriptor
and the Methods say. The stimulus code of `Stimuli.Label` (k is the whisker `whisk[k]` of
settings.mat, 9 a blank trial) agrees with the per-whisker frame lists in every field; the
descriptor does not state it. Field DNH13b/20161208/f2 has no trials for E1, E2 and E3: those
whiskers are missing from its `traceByStim`, which the descriptor does not mention. Three fields
have one dF/F movie without a `Stimuli` entry. `processed_data.mat` holds more ROIs than
`reduced_data.mat` in 21 of 36 fields (1044 against 998 in NH L2/3, 1276 against 1147 in EN L2/3);
the paper says ROIs with nuclear GCaMP were excluded, which may account for part of this. The
sponTrace blank trials are fewer than the trials Label marks blank (for example 51 of 61 in the
first field), with no stated rule.

In [ ]:
# dF/F: the stored traces against a rebuild from processed_data.mat (first movie of each field).
dchk = pd.DataFrame([dict(field=f["field"], layer=f["layer"], npSub=f["npSub"],
                          rebuild_5frame_p5_r0=f["dff_rebuild_maxdiff"], methods_4frame_p20=f["dff_methods_maxdiff"])
                     for f in F])
print("Largest absolute difference from the stored dF/F, per field (dF/F units):")
print(dchk.groupby(["layer", "npSub"])[["rebuild_5frame_p5_r0", "methods_4frame_p20"]].describe().round(4).T.to_string())
# Evoked responses rebuilt at r = 0 (L2/3) and r = 0.3 (L4) against those of the stored traces.
rows = []
for f in F:
    rw = 0.0 if f["layer"] == "L23" else 0.3
    a = np.concatenate([f["np"][rw]["ev"][w].ravel() for w in ssc10.WHISKERS
                        if f["np"][rw]["ev"][w].shape == f["ev"][w].shape])
    b = np.concatenate([f["ev"][w].ravel() for w in ssc10.WHISKERS if f["np"][rw]["ev"][w].shape == f["ev"][w].shape])
    ok = np.isfinite(a) & np.isfinite(b)
    rows.append(dict(field=f["field"], layer=f["layer"], r=rw, corr=np.corrcoef(a[ok], b[ok])[0, 1],
                     max_abs_diff=np.abs(a[ok] - b[ok]).max(), blank_match_maxdiff=f["np_blank_match_maxdiff"]))
evchk = pd.DataFrame(rows)
print("\nPer-trial evoked responses rebuilt from raw fluorescence against the stored traces:")
print(evchk.groupby("layer")[["corr", "max_abs_diff", "blank_match_maxdiff"]].describe().round(4).T.to_string())

The stored dF/F of the L2/3 fields is reproduced from `rawTimeSeries` to within 0.02 dF/F by a
5-frame moving median and F0 the 5th percentile of each movie; with the Methods' 4-frame median and
20th percentile the largest difference per field is 0.17 to 3.5. The data follow the pipeline code (`deltaF_simple.m`, as the
claims file reports it), not the Methods. Per-trial evoked responses rebuilt this way match the
stored ones (correlation 1.000, largest difference below 0.006). For L4 (`npSub` = 1) the rebuild
with r = 0.3 correlates 0.983 to 0.996 with the stored responses, but differs by up to 0.33, so the
order of filtering and subtraction in the L4 pipeline is not the one used here; this matters only
for the neuropil claim.

In [ ]:
# Stimulus timing: population mean dF/F around whisker and blank onsets, every field.
lags = F[0]["onset_lags"] / F[0]["fs"]
resp_win = (lags >= 0) & (lags < 1)
pre_win = (lags >= -0.5) & (lags < 0)
rows = []
for f in F:
    s, b = f["onset_trace_stim"], f["onset_trace_blank"]
    d = s - b
    # contrast in the 1 s after onset against the same contrast with the onsets shifted by -2 s and +2.5 s
    shift = {sh: (np.roll(d, -int(round(sh * f["fs"])))[resp_win] - np.roll(d, -int(round(sh * f["fs"])))[pre_win].mean()).mean()
             for sh in (-2.0, 0.0, 2.5)}
    rows.append(dict(field=f["field"], layer=f["layer"], stim_minus_blank_0_1s=shift[0.0],
                     shifted_m2s=shift[-2.0], shifted_p2_5s=shift[2.5], peak_lag_s=lags[np.argmax(d)]))
clock = pd.DataFrame(rows)
print(clock.round(4).to_string(index=False))
print("\nFields where the whisker-minus-blank contrast in 0-1 s exceeds both shifted contrasts:",
      int(((clock.stim_minus_blank_0_1s > clock.shifted_m2s) & (clock.stim_minus_blank_0_1s > clock.shifted_p2_5s)).sum()), "of", len(clock))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
for f in F:
    ax[0 if f["layer"] == "L23" else 1].plot(lags, f["onset_trace_stim"] - f["onset_trace_blank"], lw=0.7)
for a, t in zip(ax, ("L2/3", "L4")):
    a.axvline(0, color="k", lw=0.5); a.set_title(f"{t}: whisker minus blank, population mean dF/F"); a.set_xlabel("s from onset")
plt.tight_layout(); plt.show()
print("Fields whose difference peaks 0.3 to 1.4 s after onset:", int(clock.peak_lag_s.between(0.3, 1.4).sum()), "of", len(clock))
print("Fields where the shifted contrasts are not exceeded:", clock[~((clock.stim_minus_blank_0_1s > clock.shifted_m2s) & (clock.stim_minus_blank_0_1s > clock.shifted_p2_5s))].field.tolist())
print("Range of plotted values:", round(min((f["onset_trace_stim"] - f["onset_trace_blank"]).min() for f in F), 3),
      "to", round(max((f["onset_trace_stim"] - f["onset_trace_blank"]).max() for f in F), 3))

Stimulus timing: in 30 of 36 fields the population mean dF/F after whisker onsets exceeds that after
blank onsets more in the 1 s after onset than with the onsets shifted by -2 s or +2.5 s, and the
peak of the difference lies 0.3 to 1.4 s after onset in 28 fields. This is consistent with the
stimulus table and the dF/F sharing one clock. In the six other fields (printed above), where the
whisker-minus-blank difference of the mean over all ROIs is small, the shared clock was not
established by this test.

In [ ]:
# Barrel radius per field and barrel: ROI_pos_barrel / ROI_pos_barrel_norm (one scalar per barrel?).
rows = []
for f in F:
    ratio = np.linalg.norm(f["pos"], axis=-1) / np.linalg.norm(f["posn"], axis=-1)   # R x 9
    for j, w in enumerate(ssc10.WHISKERS):
        r = ratio[:, j]
        r = r[np.isfinite(r)]
        if len(r) == 0:
            continue
        rows.append(dict(field=f["field"], mouse=f["mouse"], layer=f["layer"], group=f["group"], barrel=w,
                         radius=np.median(r), spread=(r.max() - r.min()) / np.median(r)))
radii = pd.DataFrame(rows)
print("Relative spread (max - min) / median of the ratio over a field's ROIs, per barrel: median",
      round(radii.spread.median(), 4), ", max", round(radii.spread.max(), 4))
print("Radius (um) by layer and group, over fields and barrels:")
print(radii.groupby(["layer", "group"]).radius.describe().round(1).to_string())
# ROI positions: distance to the nearest barrel centre, inside against outside ROIsInBarrel
d_in, d_out = [], []
for f in F:
    dist = np.linalg.norm(f["pos"], axis=-1)
    distn = np.linalg.norm(f["posn"], axis=-1)
    inb = f["barrel"] != ""
    j = np.array([ssc10.WHISKERS.index(b) if b else -1 for b in f["barrel"]])
    d_in += list(distn[inb, j[inb]])
    d_out += list(distn[~inb].min(1))
print("Normalised distance to the ROI's own barrel centre, ROIs listed in ROIsInBarrel: max", round(max(d_in), 3),
      "; ROIs listed in no barrel: min distance to any centre", round(min(d_out), 3))

`ROI_pos_barrel_norm` is not `ROI_pos_barrel` divided by one radius per barrel: the ratio varies by
a median 37% over the ROIs of a field, so the normalisation uses a radius along each ROI's
direction, and no barrel radius or area is stored. ROIs listed in `ROIsInBarrel` lie within 1.27
normalised radii of their barrel's centre and unlisted ROIs at least 0.85 from any centre, which is
consistent with the barrel boundaries being drawn from histology outlines, not circles.

## Mice, fields and cells

### table1-mice-and-fields

> The population imaging used 7 NH and 7 EN Drd3-Cre mice with 11 imaging fields each, and 5 NH
> and 6 EN Scnn1a-Tg3-Cre mice with 6 and 8 imaging fields.

Folder names carry the genotype (`D` for Drd3-Cre, `Sc` for Scnn1a-Tg3-Cre) and the housing (`NH`,
`ENR`); the data descriptor's `<mouse ID>\<experiment date>\<imaging field number>` layout gives
the fields.

In [ ]:
counts = info.groupby(["layer", "group"]).agg(mice=("mouse", "nunique"), fields=("field", "size"))
counts.index = [f"{'Drd3-Cre' if l == 'L23' else 'Scnn1a-Tg3-Cre'} {g}" for l, g in counts.index]
counts["paper_mice"] = [7, 7, 6, 5]
counts["paper_fields"] = [11, 11, 8, 6]
print(counts.to_string())
print("\nMice imaged on two dates:", sorted(info.groupby("mouse").field.apply(lambda s: s.str.split("/").str[1].nunique()).loc[lambda s: s > 1].index))

**Result: reproduced.** The data set holds 7 NH and 7 EN Drd3-Cre mice with 11 fields each, and 5
NH and 6 EN Scnn1a-Cre mice with 6 and 8 fields, as Table 1 gives.

### Responsiveness

The Methods' test (claims file, opening comment) is run on every ROI: a one-sided permutation test
of the difference in mean evoked dF/F between whisker and blank trials, 10,000 shuffles,
Benjamini-Hochberg over the nine whiskers at 0.05. The variant is the data set's stored
`permTestResults`, corrected the same way and without ROIs whose traces contain NaN, as the data
set's sample code (`find_sigROIs.m`) does.

In [ ]:
rng = np.random.default_rng(101)
t0 = time.time()
ssc10.responsiveness(F, rng)
print(f"responsiveness tests: {time.time() - t0:.0f} s")
rows = []
for f in F:
    inb = f["barrel"] != ""
    crow = np.isin(f["barrel"], ["c1", "c2", "c3"])
    rows.append(dict(layer=f["layer"], group=f["group"], total=f["n_rois"], total_raw=f["n_rois_raw"],
                     resp_methods=f["resp_methods"].sum(), resp_methods_barrel=(f["resp_methods"] & inb).sum(),
                     resp_methods_C123=(f["resp_methods"] & crow).sum(),
                     resp_stored=f["resp_stored"].sum(), resp_stored_barrel=(f["resp_stored"] & inb).sum(),
                     resp_stored_C123=(f["resp_stored"] & crow).sum()))
cc = pd.DataFrame(rows).groupby(["layer", "group"]).sum()
cc["frac_resp_methods"] = (cc.resp_methods / cc.total).round(3)
print(cc.to_string())
agree = np.concatenate([f["resp_methods"] == f["resp_stored"] for f in F])
print(f"\nROIs on which the Methods' test and the stored p-values agree: {agree.mean():.3f}")
print("Stored p-values: smallest value", min(np.nanmin(f["p_stored"]) for f in F),
      "; Methods' test smallest possible 1/10001 =", round(1 / 10001, 6))

### l23-cell-counts

> In L2/3 of Drd3-Cre mice, NH fields held 998 cells of which 475 were whisker-responsive, and EN
> fields 1257 cells of which 874 were responsive.

In [ ]:
l23 = cc.loc["L23"]
tab = pd.DataFrame({
    "paper": [998, 475, 1257, 874],
    "tolerance": [99.8, 47.5, 125.7, 87.4],
    "total (traceByStim ROIs)": [l23.loc["NH", "total"], None, l23.loc["EN", "total"], None],
    "A: Methods test, whole field": [None, l23.loc["NH", "resp_methods"], None, l23.loc["EN", "resp_methods"]],
    "B: Methods test, over a barrel": [None, l23.loc["NH", "resp_methods_barrel"], None, l23.loc["EN", "resp_methods_barrel"]],
    "A: stored p, whole field": [None, l23.loc["NH", "resp_stored"], None, l23.loc["EN", "resp_stored"]],
    "B: stored p, over a barrel": [None, l23.loc["NH", "resp_stored_barrel"], None, l23.loc["EN", "resp_stored_barrel"]],
}, index=["NH total", "NH responsive", "EN total", "EN responsive"])
print(tab.to_string())
print("\nROIs in processed_data.mat (before the exclusions of reduced_data.mat): NH", l23.loc["NH", "total_raw"], ", EN", l23.loc["EN", "total_raw"])
print("Fraction responsive (Methods test, whole field): NH", l23.loc["NH", "frac_resp_methods"], ", EN", l23.loc["EN", "frac_resp_methods"])

**Result: reproduced.** Under reading A with the Methods' test, NH fields hold 998 ROIs of which 472
are responsive, and EN fields 1147 of which 845 (paper 998, 475, 1257, 874); all four are within
tolerance, the EN total by 110 of an allowed 125.7. Reading A with the stored p-values gives 474 and
828, also within tolerance; reading B gives 376 and 635, outside. The responsive fraction is higher
in EN (0.737 against 0.473).

### l4-cell-counts

> In L4 of Scnn1a-Cre mice, NH fields held 322 cells of which 107 were responsive, and EN fields 586
> cells of which 181 were responsive.

In [ ]:
l4 = cc.loc["L4"]
tab = pd.DataFrame({
    "paper": [322, 107, 586, 181],
    "tolerance": [32.2, 10.7, 58.6, 18.1],
    "total (traceByStim ROIs)": [l4.loc["NH", "total"], None, l4.loc["EN", "total"], None],
    "total (processed_data ROIs)": [l4.loc["NH", "total_raw"], None, l4.loc["EN", "total_raw"], None],
    "A: Methods test, whole field": [None, l4.loc["NH", "resp_methods"], None, l4.loc["EN", "resp_methods"]],
    "B: Methods test, C1-C3": [None, l4.loc["NH", "resp_methods_C123"], None, l4.loc["EN", "resp_methods_C123"]],
    "A: stored p, whole field": [None, l4.loc["NH", "resp_stored"], None, l4.loc["EN", "resp_stored"]],
    "B: stored p, C1-C3": [None, l4.loc["NH", "resp_stored_C123"], None, l4.loc["EN", "resp_stored_C123"]],
}, index=["NH total", "NH responsive", "EN total", "EN responsive"])
print(tab.to_string())

**Result: not reproduced.** The NH L4 fields hold 362 ROIs (paper 322, tolerance 32.2) and EN 615
(586, within). Responsive ROIs in C1-C3 are 95 NH and 162 EN by the Methods' test (paper 107 and
181; tolerances 10.7 and 18.1), 85 and 151 by the stored p-values; over the whole field 228 and 239,
or 219 and 205. No reading puts all four counts within tolerance.

### fig1f-gcamp-matches-spikes

> Whisker receptive fields measured from GCaMP6s dF/F were highly similar to those from
> simultaneously recorded spikes in 9 of 10 L2/3 neurons.

The data set holds no cell-attached recordings: every field above is a Drd3-Cre or Scnn1a-Cre
population imaging field, and the data descriptor lists only the three MATLAB files per field.
This claim is blocked.

## Per-ROI measures

For each responsive ROI: the evoked response per whisker as the median over trials (the receptive
field of the Methods) and as the mean; the median z-scored to the blank trials, centred on the
blank mean (`z`, the claims file's reading) and on the blank median (`zmed`, the variant; it is what
the data set's sample code `plotPointRep_CRCNS.m` computes); the best whisker (BW, largest mean;
`best_med` by the median); equivalent best whiskers under readings E1, E2 (10,000 shuffles) and
E3 (medians, 1,000 shuffles); the distance in um to each of the nine barrel centres. L2/3 uses all
responsive ROIs of a field (reading A) unless a cell says otherwise; L4 uses responsive ROIs in
barrels C1 to C3 (Methods), all responsive L4 ROIs as the variant.

In [ ]:
t0 = time.time()
T, A = ssc10.roi_table(F, "methods", np.random.default_rng(202), ebw=True, ebw_med=True)
print(f"ROI table (Methods responsiveness): {len(T)} ROIs, {time.time() - t0:.0f} s")
TS, AS = ssc10.roi_table(F, "stored", np.random.default_rng(203), ebw=True, ebw_med=False)
print(f"ROI table (stored p-values): {len(TS)} ROIs")
W = np.array(ssc10.WHISKERS)
for tb in (T, TS):
    tb["inb"] = tb.barrel != ""
    tb["c123"] = tb.barrel.isin(["c1", "c2", "c3"])
print(T.groupby(["layer", "group"]).agg(n=("roi", "size"), in_barrel=("inb", "sum"), c123=("c123", "sum")).to_string())
print("\nz (blank-mean centred), range over all ROIs and whiskers:", np.round(np.nanmin(A["z"]), 2), "to", np.round(np.nanmax(A["z"]), 2))
print("BW by mean and by median agree for", round((T.best == T.best_med).mean(), 3), "of ROIs")
print("Mean number of eBWs including the BW: E1", A["e1"].sum(1).mean().round(2), ", E2", A["e2"].sum(1).mean().round(2),
      ", E3", A["e3"].sum(1).mean().round(2))

## Point representation (Figure 2)

Each responsive ROI enters once per reference whisker, at its distance to that whisker's barrel
centre, with its z-scored median response to that whisker (claims file, opening comment). Bins are
75 um (Figure 2A legend; `binEdges=0:75:1050` in the data set's sample code). EN against NH per
bin by the two-sided permutation test of the difference in means (10,000 shuffles).

In [ ]:
BINS = np.arange(0, 1051, 75)


def pointrep(tb, ar, layer, zkey="z", sel=None, rng=None, bins=BINS):
    m = (tb.layer == layer).values
    if sel is not None:
        m = m & sel
    out = []
    for b0, b1 in zip(bins[:-1], bins[1:]):
        r = {}
        for g in ("NH", "EN"):
            mm = m & (tb.group == g).values
            d, z = ar["dist"][mm], ar[zkey][mm]
            inb = (d >= b0) & (d < b1) & np.isfinite(z)
            r[g] = z[inb]
            r[g + "_mice"] = np.repeat(tb.mouse.values[mm][:, None], 9, 1)[inb]
        diff, p = ssc10.perm_test_means(r["EN"], r["NH"], rng)
        out.append(dict(bin=f"{b0}-{b1}", n_NH=len(r["NH"]), NH=r["NH"].mean(), sem_NH=ssc10.sem(r["NH"]),
                        n_EN=len(r["EN"]), EN=r["EN"].mean(), sem_EN=ssc10.sem(r["EN"]), diff=diff, p=p))
    return pd.DataFrame(out).set_index("bin")


L23_READINGS = {}
rng = np.random.default_rng(301)
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for roi_reading, sel in (("A", None), ("B", tb.inb.values)):
        for zkey in ("z", "zmed"):
            L23_READINGS[(resp, roi_reading, zkey)] = pointrep(tb, ar, "L23", zkey, None if sel is None else sel, rng)
main = L23_READINGS[("methods", "A", "z")]
print("L2/3, Methods responsiveness, reading A, z centred on the blank mean:")
print(main.round(4).to_string())

### fig2a-l23-point-representation-stronger

> In L2/3, EN raised the mean response to a reference whisker in and just around its column, at
> 0-75 um (NH 0.59, EN 0.77), 150-225 um (0.53, 0.60) and 225-300 um (0.41, 0.51) from the column
> centre.

In [ ]:
EXP = {"0-75": (0.59, 0.06, 0.77, 0.04), "150-225": (0.53, 0.02, 0.60, 0.02), "225-300": (0.41, 0.02, 0.51, 0.01)}
rows = []
for key, pr in L23_READINGS.items():
    rec = dict(responsive=key[0], rois=key[1], zscore=key[2])
    tests, within = [], []
    for b, (nh, tnh, en, ten) in EXP.items():
        rec[f"NH {b}"] = round(pr.loc[b, "NH"], 3); rec[f"EN {b}"] = round(pr.loc[b, "EN"], 3); rec[f"p {b}"] = pr.loc[b, "p"]
        tests.append(pr.loc[b, "diff"] > 0 and pr.loc[b, "p"] < 0.05)
        within += [abs(pr.loc[b, "NH"] - nh) <= tnh, abs(pr.loc[b, "EN"] - en) <= ten]
    rec["tests_hold"] = int(sum(tests)); rec["values_within"] = int(sum(within))
    rows.append(rec)
fig2a = pd.DataFrame(rows)
print(fig2a.to_string(index=False))
print("\nCounts in the 0-75 um bin (paper, Figure 3 S1: 58 NH ROI entries); 225-300 um (paper 406):",
      main.loc["0-75", "n_NH"], main.loc["225-300", "n_NH"])
fig, ax = plt.subplots(figsize=(6, 3.5))
x = BINS[:-1] + 37.5
for g, c in (("NH", "k"), ("EN", "tab:red")):
    ax.errorbar(x, main[g], main["sem_" + g], color=c, label=g, marker="o", ms=3)
ax.set_xlabel("distance from reference column centre (um)"); ax.set_ylabel("z-scored response"); ax.legend()
ax.set_title("L2/3, reading A, Methods responsiveness"); plt.tight_layout(); plt.show()
print("Plotted means range from", round(main[["NH", "EN"]].min().min(), 3), "to", round(main[["NH", "EN"]].max().max(), 3))

In [ ]:
# By mouse: mean z per mouse in the three bins (reading A, Methods responsiveness, z on the blank mean).
m = (T.layer == "L23").values
rows = []
for b in ("0-75", "150-225", "225-300", "375-450", "525-600"):
    b0, b1 = map(int, b.split("-"))
    for i in np.where(m)[0]:
        for j in range(9):
            d = A["dist"][i, j]
            if b0 <= d < b1 and np.isfinite(A["z"][i, j]):
                rows.append(dict(bin=b, group=T.group.iat[i], mouse=T.mouse.iat[i], z=A["z"][i, j]))
bym = pd.DataFrame(rows).groupby(["bin", "group", "mouse"]).z.agg(["mean", "size"]).round(3)
print(bym.unstack("bin").to_string())

**Result: reproduced, under one reading.** With the stored p-values for responsiveness, reading A
and the z-score centred on the blank median (the data set's sample code computes this), EN is higher
in the three bins (NH 0.570, 0.515, 0.404; EN 0.758, 0.597, 0.504; p = 0.013, 0.0018, 0.0002) and all
six values are within tolerance. With the Methods' test for responsiveness, EN is higher in all
three bins but significantly only at 225-300 um (p = 0.32, 0.081, 0.0022 for the blank-mean z), so
the claim holds only with the stored responsiveness. The 0-75 um NH bin holds 48 entries by the
Methods' test and 57 by the stored p-values (Figure 3 S1: 58).

### fig2a-l23-flanks-suppressed

> In L2/3, EN lowered mean responses to the reference whisker at the flanks of the point
> representation, 375 to 750 um from the column centre.

In [ ]:
FL = ["375-450", "450-525", "525-600", "600-675", "675-750"]
rows = []
for key, pr in L23_READINGS.items():
    rec = dict(responsive=key[0], rois=key[1], zscore=key[2])
    for b in FL:
        rec[f"EN-NH {b}"] = round(pr.loc[b, "diff"], 3); rec[f"p {b}"] = pr.loc[b, "p"]
    rec["bins EN lower p<0.05"] = int(sum((pr.loc[b, "diff"] < 0) and (pr.loc[b, "p"] < 0.05) for b in FL))
    rows.append(rec)
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** EN is lower than NH in all five bins from 375 to 750 um under
every reading, significantly in at most three of them (375-450, 525-600 and 600-675 um under the
Methods' test, reading A, blank-mean z: p = 0.0008, 0.0006, 0.0072; 450-525 and 675-750 um p = 0.22
and 0.10).

### fig2b-2d-point-representation

> Two-dimensional maps of the L2/3 point representation, with fields rotated to align the arc axis,
> show the same EN enhancement in the home column and reduction some distance away.

The claim is judged on the EN minus NH mean response over ROI-whisker entries inside the average
barrel radius of the reference column, and beyond 375 um (to 750 um), as the criterion says. The
mean radius is the mean over L2/3 fields and barrels of the radius from `ROI_pos_barrel_norm`
(data check). The 2D map is drawn with axes along the row and arc of each field, from the barrel
centres (`ROI_pos_barrel` relative to two barrels gives the vector between their centres).

In [ ]:
r_mean = radii[radii.layer == "L23"].radius.mean()
print(f"Mean L2/3 barrel radius from the data set's normalised positions: {r_mean:.1f} um")
rng = np.random.default_rng(302)
rows = []
for key in L23_READINGS:
    resp, rr, zkey = key
    tb, ar = (T, A) if resp == "methods" else (TS, AS)
    m = (tb.layer == "L23").values & (tb.inb.values if rr == "B" else True)
    res = {}
    for name, lo, hi in (("inside", 0, r_mean), ("375-750", 375, 750)):
        vals = {}
        for g in ("NH", "EN"):
            mm = m & (tb.group == g).values
            d, z = ar["dist"][mm], ar[zkey][mm]
            vals[g] = z[(d >= lo) & (d < hi) & np.isfinite(z)]
        diff, p = ssc10.perm_test_means(vals["EN"], vals["NH"], rng)
        res[name] = (diff, p, len(vals["NH"]), len(vals["EN"]))
    rows.append(dict(responsive=resp, rois=rr, zscore=zkey,
                     inside_EN_minus_NH=round(res["inside"][0], 3), inside_p=res["inside"][1], inside_n=res["inside"][2:],
                     far_EN_minus_NH=round(res["375-750"][0], 3), far_p=res["375-750"][1], far_n=res["375-750"][2:]))
print(pd.DataFrame(rows).to_string(index=False))


def field_axes(f):
    """Unit vectors along the row (arc number increasing) and the arc (C to E) in a field's image frame."""
    c = {}
    for j, w in enumerate(ssc10.WHISKERS):
        ok = np.isfinite(f["pos"][:, j, 0]) & np.isfinite(f["pos"][:, 0, 0])
        c[w] = np.nanmean(f["pos"][ok, 0, :] - f["pos"][ok, j, :], 0)   # centre of barrel w relative to C1
    row = np.nanmean([c[r + "2"] - c[r + "1"] for r in "cde"] + [c[r + "3"] - c[r + "2"] for r in "cde"], 0)
    arc = np.nanmean([c["d" + a] - c["c" + a] for a in "123"] + [c["e" + a] - c["d" + a] for a in "123"], 0)
    return row / np.linalg.norm(row), arc / np.linalg.norm(arc), c


AXES = {f["field"]: field_axes(f) for f in F}
ang = [np.degrees(np.arccos(abs(np.dot(a[0], a[1])))) for a in AXES.values()]
print(f"Angle between the row and arc axes over fields: {min(ang):.0f} to {max(ang):.0f} degrees")
# 2D map: each L2/3 ROI-whisker entry at its (along-row, along-arc) position from the reference centre
fig, axs = plt.subplots(1, 3, figsize=(13, 4))
grid = np.arange(-750, 751, 150)
maps = {}
for g in ("NH", "EN"):
    xs, ys, zs = [], [], []
    for i in np.where((T.layer == "L23").values & (T.group == g).values)[0]:
        f = F[T.fi.iat[i]]
        rowv, arcv, _ = AXES[f["field"]]
        for j in range(9):
            p = f["pos"][T.row.iat[i], j]
            xs.append(p @ rowv); ys.append(p @ arcv); zs.append(A["z"][i, j])
    xs, ys, zs = map(np.array, (xs, ys, zs))
    ok = np.isfinite(zs)
    H, _, _ = np.histogram2d(ys[ok], xs[ok], bins=[grid, grid], weights=zs[ok])
    N, _, _ = np.histogram2d(ys[ok], xs[ok], bins=[grid, grid])
    maps[g] = np.where(N >= 5, H / np.maximum(N, 1), np.nan)
for a, (t, mp) in zip(axs, (("NH", maps["NH"]), ("EN", maps["EN"]), ("EN - NH", maps["EN"] - maps["NH"]))):
    im = a.imshow(mp, extent=[-750, 750, 750, -750], cmap="viridis" if t != "EN - NH" else "RdBu_r")
    a.set_title(f"L2/3 {t}"); a.set_xlabel("along row (um, + = higher arc)"); a.set_ylabel("along arc (um, + = ventral row)")
    plt.colorbar(im, ax=a)
plt.tight_layout(); plt.show()
print("Map ranges: NH", np.nanmin(maps["NH"]).round(3), "to", np.nanmax(maps["NH"]).round(3), "; EN",
      np.nanmin(maps["EN"]).round(3), "to", np.nanmax(maps["EN"]).round(3), "; 150 um squares with fewer than 5 entries left blank")

**Result: reproduced.** Inside the mean barrel radius (151 um) EN minus NH is positive under every
reading (+0.070 to +0.103, p = 0.002 to 0.031), and from 375 to 750 um it is negative under every
reading (-0.022 to -0.042, p = 0.0001 to 0.0091). The 2D maps are shown for reading A.

### fig2c-l4-point-representation-weaker

> In L4, EN reduced mean whisker-evoked responses in most bins within 375 um of the reference column
> centre, including 0-75 um (NH 1.11, EN 0.86).

In [ ]:
rng = np.random.default_rng(303)
L4_READINGS = {}
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for sel_name, sel in (("C1-C3", tb.c123.values), ("all L4", None)):
        for zkey in ("z", "zmed"):
            L4_READINGS[(resp, sel_name, zkey)] = pointrep(tb, ar, "L4", zkey, sel, rng)
print("L4, Methods responsiveness, C1-C3, z centred on the blank mean:")
print(L4_READINGS[("methods", "C1-C3", "z")].iloc[:8].round(4).to_string())
NEAR = ["0-75", "75-150", "150-225", "225-300", "300-375"]
rows = []
for key, pr in L4_READINGS.items():
    rec = dict(responsive=key[0], rois=key[1], zscore=key[2], NH_0_75=round(pr.loc["0-75", "NH"], 3),
               EN_0_75=round(pr.loc["0-75", "EN"], 3), p_0_75=pr.loc["0-75", "p"],
               n_0_75=(pr.loc["0-75", "n_NH"], pr.loc["0-75", "n_EN"]))
    low = [(pr.loc[b, "diff"] < 0) and (pr.loc[b, "p"] < 0.05) for b in NEAR]
    rec["bins<375 EN lower p<0.05"] = int(sum(low)); rec["which"] = ",".join(b for b, l in zip(NEAR, low) if l); rec["0-75 among them"] = bool(low[0])
    rec["values within"] = int(abs(pr.loc["0-75", "NH"] - 1.11) <= 0.13) + int(abs(pr.loc["0-75", "EN"] - 0.86) <= 0.06)
    rows.append(rec)
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** At 0-75 um the L4 means are NH 0.972 and EN 0.926 (C1-C3, Methods'
test; p = 0.72); no reading makes EN significantly lower there (p = 0.19 to 0.95). EN is
significantly lower in one to three of the five bins below 375 um, depending on the reading (in
the Methods' C1-C3 reading only at 150-225 um, p = 0.0027). The stored p-values with C1-C3 give NH
1.101 at 0-75 um, within tolerance, and EN 0.921, just outside it, with p = 0.19.

### fig2c-l4-sharper-than-l23

> In NH mice the point representation was sharper in L4 than in L2/3.

Readings of the criterion: (a) the mean at 225-300 um over the mean at 0-75 um; (b) the first
distance bin whose mean falls below half the 0-75 um value. NH profiles.

In [ ]:
def sharp(pr):
    nh = pr["NH"]
    ratio = nh["225-300"] / nh["0-75"]
    below = [b for b in nh.index if nh[b] < 0.5 * nh["0-75"]]
    return ratio, below[0] if below else None


rows = []
for (resp, rr, zkey), pr23 in L23_READINGS.items():
    for sel in ("C1-C3", "all L4"):
        pr4 = L4_READINGS[(resp, sel, zkey)]
        (r23, h23), (r4, h4) = sharp(pr23), sharp(pr4)
        rows.append(dict(responsive=resp, l23_rois=rr, l4_rois=sel, zscore=zkey, ratio_L23=round(r23, 3), ratio_L4=round(r4, 3),
                         half_L23=h23, half_L4=h4,
                         both_sharper_L4=(r4 < r23) and (int(h4.split("-")[0]) < int(h23.split("-")[0]))))
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** In NH mice the 225-300 um response is 0.24 to 0.39 of the 0-75 um response
in L4 against 0.58 to 0.71 in L2/3, and the profile falls below half its central value at 225-300 um
in L4 against 375-450 or 450-525 um in L2/3, under every reading.

### figs2-1-column-frame

> In a column-based reference frame, EN raised the mean L2/3 response to a reference whisker in its
> home column, sharpened the point representation mainly across rows, and reduced its spatial bias
> toward adjacent higher-arc columns; in L4 EN weakened home-column responses.

Positions in units of the reference barrel's radius along the ROI's direction
(`ROI_pos_barrel_norm`), ROIs within 5 radii; home column below 1. Directions are taken from each
field's row and arc axes (the vectors between barrel centres). Across rows means within 45 degrees
of the arc axis; across arcs, within 45 degrees of the row axis.

In [ ]:
def frame_entries(tb, ar, layer, sel=None, zkey="z"):
    m = (tb.layer == layer).values
    if sel is not None:
        m = m & sel
    rows = []
    for i in np.where(m)[0]:
        f = F[tb.fi.iat[i]]
        rowv, arcv, _ = AXES[f["field"]]
        for j in range(9):
            p = f["pos"][tb.row.iat[i], j]
            dn = ar["distn"][i, j]
            if not np.isfinite(dn) or dn > 5:
                continue
            x, y = p @ rowv, p @ arcv
            ang = np.degrees(np.arctan2(y, x))   # 0 = toward higher arc, 90 = toward the next row
            rows.append((tb.group.iat[i], dn, ang, ar[zkey][i, j]))
    return pd.DataFrame(rows, columns=["group", "dn", "ang", "z"])


rng = np.random.default_rng(401)
res = []
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for zkey in ("z", "zmed"):
        e23 = frame_entries(tb, ar, "L23", None, zkey)
        e4 = frame_entries(tb, ar, "L4", tb.c123.values, zkey)
        rec = dict(responsive=resp, zscore=zkey)
        for lay, e in (("L23", e23), ("L4", e4)):
            h = e[e.dn < 1]
            d, p = ssc10.perm_test_means(h[h.group == "EN"].z, h[h.group == "NH"].z, rng)
            rec[f"{lay} home NH"] = round(h[h.group == "NH"].z.mean(), 3); rec[f"{lay} home EN"] = round(h[h.group == "EN"].z.mean(), 3)
            rec[f"{lay} home p"] = p
        ring = e23[(e23.dn >= 1) & (e23.dn < 2)]
        a = ring.ang.abs()
        across_rows = ring[(a > 45) & (a < 135)]
        across_arcs = ring[(a <= 45) | (a >= 135)]
        plus_arc = ring[a <= 45]
        minus_arc = ring[a >= 135]
        g = lambda df, grp: df[df.group == grp].z.mean()
        rec["rows EN-NH"] = round(g(across_rows, "EN") - g(across_rows, "NH"), 3)
        rec["arcs EN-NH"] = round(g(across_arcs, "EN") - g(across_arcs, "NH"), 3)
        rec["bias NH (+arc - -arc)"] = round(g(plus_arc, "NH") - g(minus_arc, "NH"), 3)
        rec["bias EN (+arc - -arc)"] = round(g(plus_arc, "EN") - g(minus_arc, "EN"), 3)
        rec["n ring rows/arcs"] = (len(across_rows), len(across_arcs))
        res.append(rec)
print(pd.DataFrame(res).to_string(index=False))

**Result: not reproduced.** The L2/3 home-column response is higher in EN under every reading,
significantly with the stored p-values (0.602 against 0.697, p = 0.0059; Methods' test p = 0.17 and
0.065). The L4 home-column response is not lower in EN at p < 0.05 under any reading (p = 0.27 to
0.94). The two readings of the shape hold: at 1-2 radii EN minus NH is -0.048 across rows and
+0.050 across arcs, and the +1 minus -1 arc bias is +0.240 in NH and -0.125 in EN (Methods' test,
blank-mean z). The criterion's partial outcome needs both home-column tests.

### figs2-2-barrel-size-unchanged

> Enrichment did not change the radius of L4 barrels, in Drd3-Cre mice (NH 151 um, EN 145 um) or in
> Scnn1a-Cre mice (NH 118 um, EN 124 um).

The data set holds no barrel areas. The radius used is the median over a field's ROIs of
`ROI_pos_barrel` / `ROI_pos_barrel_norm` for each barrel, which is the barrel's radius along the
ROIs' directions (a stated approximation, as the criterion allows), averaged over a mouse's fields.
Each mouse's mean is over its nine barrels; the ANOVA is on the per-barrel radii per mouse.

In [ ]:
rb = radii.groupby(["layer", "group", "mouse", "barrel"]).radius.mean().reset_index()
per_mouse = rb.groupby(["layer", "group", "mouse"]).radius.agg(["mean", "size"]).reset_index()
print(per_mouse.round(1).to_string(index=False))
EXPR = {("L23", "NH"): (151, 7.2), ("L23", "EN"): (145, 4.6), ("L4", "NH"): (118, 6.6), ("L4", "EN"): (124, 6.9)}
for lay in ("L23", "L4"):
    d = rb[rb.layer == lay]
    model = smf.ols("radius ~ C(group) + C(barrel)", data=d).fit()
    an = anova_lm(model, typ=2)
    pm = per_mouse[per_mouse.layer == lay]
    for g in ("NH", "EN"):
        v = pm[pm.group == g]["mean"]
        e, tol = EXPR[(lay, g)]
        print(f"{'Drd3-Cre' if lay == 'L23' else 'Scnn1a-Cre'} {g}: {v.mean():.1f} +/- {ssc10.sem(v):.1f} um (SEM), n = {len(v)} mice;"
              f" paper {e} +/- {tol}; within: {abs(v.mean() - e) <= tol}")
    print(f"  two-way ANOVA, enrichment factor: F = {an.loc['C(group)', 'F']:.2f}, p = {an.loc['C(group)', 'PR(>F)']:.4f};"
          f" barrel factor p = {an.loc['C(barrel)', 'PR(>F)']:.4g}")

**Result: partially reproduced.** The two-way ANOVA finds no enrichment effect in either genotype
(Drd3-Cre p = 0.20, Scnn1a-Cre p = 0.27). The mouse means are Drd3-Cre NH 148.2 and EN 155.8 um,
Scnn1a-Cre NH 119.1 and EN 126.5 um; three are within tolerance, Drd3-Cre EN is 10.8 um above 145
(tolerance 4.6). The radii are taken from the normalised ROI positions, along the ROIs' directions,
not from barrel areas, which the data set does not hold.

## Tuning (Figure 3)

An entry is a responsive ROI and a reference whisker; its distance is to that whisker's barrel
centre. Tuned by BW means the BW is the reference whisker; by eBW, the reference whisker is among
the eBWs.

In [ ]:
def tuning_entries(tb, ar, layer, sel=None):
    m = (tb.layer == layer).values
    if sel is not None:
        m = m & sel
    idx = np.where(m)[0]
    bw = np.array([ssc10.WHISKERS.index(b) for b in tb.best.values[idx]])
    bwm = np.array([ssc10.WHISKERS.index(b) for b in tb.best_med.values[idx]])
    out = []
    for j in range(9):
        out.append(pd.DataFrame(dict(group=tb.group.values[idx], mouse=tb.mouse.values[idx], ref=j,
                                     dist=ar["dist"][idx, j], bw=bw == j, bw_med=bwm == j,
                                     e1=ar["e1"][idx, j], e2=ar["e2"][idx, j], e3=ar["e3"][idx, j],
                                     has=np.isfinite(ar["z"][idx, j]))))
    e = pd.concat(out)
    return e[e.has & np.isfinite(e.dist)]


def bin_of(d, bins=BINS):
    k = np.digitize(d, bins) - 1
    return np.array([f"{bins[i]}-{bins[i + 1]}" if 0 <= i < len(bins) - 1 else "far" for i in k])


ENT = {}
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for rr, sel in (("A", None), ("B", tb.inb.values)):
        e = tuning_entries(tb, ar, "L23", sel)
        e["bin"] = bin_of(e.dist.values)
        ENT[("L23", resp, rr)] = e
    for rr, sel in (("C1-C3", tb.c123.values), ("all L4", None)):
        e = tuning_entries(tb, ar, "L4", sel)
        e["bin"] = bin_of(e.dist.values)
        ENT[("L4", resp, rr)] = e
print("Entries per layer, responsiveness and ROI reading:", {k: len(v) for k, v in ENT.items()})

### figs3-1-l23-salt-and-pepper

> In L2/3 of NH mice, tuning is intermixed within columns; 53% of responsive cells within 75 um of
> a column centre had the CW as best whisker, falling to 16% at 225-300 um and 1% at 525-600 um, and
> by equivalent best whiskers 67%, 43% and 10%.

E3 (medians) is computed only for the Methods' responsiveness.

In [ ]:
EXP31 = {"bw": {"0-75": (53, 5.3), "225-300": (16, 1.6), "525-600": (1, 1)},
         "ebw": {"0-75": (67, 6.7), "225-300": (43, 4.3), "525-600": (10, 1)}}
rows = []
for (lay, resp, rr), e in ENT.items():
    if lay != "L23":
        continue
    nh = e[e.group == "NH"]
    for bwk in ("bw", "bw_med"):
        for ek in ("e1", "e2", "e3"):
            if ek == "e3" and resp == "stored":
                continue
            rec = dict(responsive=resp, rois=rr, bw=bwk, ebw=ek)
            okb, oke = [], []
            for b in ("0-75", "225-300", "525-600"):
                s = nh[nh.bin == b]
                fb, fe = 100 * s[bwk].mean(), 100 * s[ek].mean()
                rec[f"n {b}"] = len(s); rec[f"BW% {b}"] = round(fb, 1); rec[f"eBW% {b}"] = round(fe, 1)
                okb.append(abs(fb - EXP31["bw"][b][0]) <= EXP31["bw"][b][1])
                oke.append(abs(fe - EXP31["ebw"][b][0]) <= EXP31["ebw"][b][1])
            rec["BW within"] = int(sum(okb)); rec["eBW within"] = int(sum(oke))
            rec["n within (58, 406)"] = int(abs(rec["n 0-75"] - 58) <= 5.8) + int(abs(rec["n 225-300"] - 406) <= 40.6)
            rows.append(rec)
s31 = pd.DataFrame(rows)
print(s31.to_string(index=False))

**Result: partially reproduced.** By BW the NH fractions are within tolerance: 54.2%, 15.0% and 1.7%
(Methods' test, reading A, BW by mean; paper 53, 16, 1), and 50.9%, 16.7% and 1.5% with the stored
p-values and the BW by median, where the bin counts are also within tolerance (57 and 406; the
Methods' test gives 48 in the 0-75 um bin). By eBW no reading comes near: 89.6% to 96.5% at 0-75 um,
55.8% to 67.7% at 225-300 um and 33.2% to 44.8% at 525-600 um (paper 67, 43, 10). Fractions fall with
distance under every reading.

### discussion-l23-nh-in-column-tuning

> In NH L2/3, about 48% of cells located in a column responded maximally to the CW and 65% had the CW
> among their equivalent best whiskers.

In [ ]:
rows = []
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    m = (tb.layer == "L23").values & (tb.group == "NH").values & tb.inb.values
    idx = np.where(m)[0]
    cw = np.array([ssc10.WHISKERS.index(b) for b in tb.barrel.values[idx]])
    d = pd.DataFrame(dict(col=tb.barrel.values[idx], bw=tb.best.values[idx] == tb.barrel.values[idx],
                          bw_med=tb.best_med.values[idx] == tb.barrel.values[idx],
                          e1=ar["e1"][idx, cw], e2=ar["e2"][idx, cw], e3=ar["e3"][idx, cw]))
    for bwk in ("bw", "bw_med"):
        for ek in ("e1", "e2", "e3"):
            if ek == "e3" and resp == "stored":
                continue
            pooled_b, pooled_e = 100 * d[bwk].mean(), 100 * d[ek].mean()
            percol = d.groupby("col")[[bwk, ek]].mean() * 100
            rows.append(dict(responsive=resp, bw=bwk, ebw=ek, n=len(d), BW_pooled=round(pooled_b, 1), eBW_pooled=round(pooled_e, 1),
                             BW_per_column=round(percol[bwk].mean(), 1), eBW_per_column=round(percol[ek].mean(), 1),
                             columns=len(percol)))
disc = pd.DataFrame(rows)
disc["pooled within"] = (abs(disc.BW_pooled - 48) <= 4.8) & (abs(disc.eBW_pooled - 65) <= 6.5)
disc["per-column within"] = (abs(disc.BW_per_column - 48) <= 4.8) & (abs(disc.eBW_per_column - 65) <= 6.5)
print(disc.to_string(index=False))

**Result: partially reproduced.** Pooled over the 376 NH L2/3 ROIs over a barrel, 48.9% have the CW
as BW (paper 48; 45.6% to 50.0% over the BW and responsiveness readings), but 84.8% to 88.1% have
the CW among the eBWs (paper 65). The mean over columns of the per-column fraction gives 34.4% to
37.6% and 82.4% to 86.4%, neither within tolerance.

### figs3-1-l4-nh-tuning

> In NH L4, 63% (30/48) of cells within 75 um of a column centre had the CW as BW and 73% (35/48) had
> it among eBWs, not different from L2/3; but the fraction tuned to a whisker fell off more sharply
> with distance in L4, significantly at 150-375 um.

In [ ]:
rows = []
for resp in ("methods", "stored"):
    l23 = ENT[("L23", resp, "A")]
    l23 = l23[l23.group == "NH"]
    for rr in ("C1-C3", "all L4"):
        l4 = ENT[("L4", resp, rr)]
        l4 = l4[l4.group == "NH"]
        for ek in ("e1", "e2", "e3"):
            if ek == "e3" and resp == "stored":
                continue
            rec = dict(responsive=resp, l4_rois=rr, ebw=ek)
            s4, s23 = l4[l4.bin == "0-75"], l23[l23.bin == "0-75"]
            rec["n L4 0-75"] = len(s4)
            for meas in ("bw", ek):
                k4, k23 = int(s4[meas].sum()), int(s23[meas].sum())
                p = stats.fisher_exact([[k4, len(s4) - k4], [k23, len(s23) - k23]])[1]
                rec[f"{'BW' if meas == 'bw' else 'eBW'} L4 0-75 %"] = round(100 * k4 / len(s4), 1)
                rec[f"{'BW' if meas == 'bw' else 'eBW'} L23 0-75 %"] = round(100 * k23 / len(s23), 1)
                rec[f"{'BW' if meas == 'bw' else 'eBW'} Fisher p"] = round(p, 4)
            for meas in ("bw", ek):
                sig = []
                for b in ("150-225", "225-300", "300-375"):
                    a4, a23 = l4[l4.bin == b], l23[l23.bin == b]
                    k4, k23 = int(a4[meas].sum()), int(a23[meas].sum())
                    p = stats.fisher_exact([[k4, len(a4) - k4], [k23, len(a23) - k23]])[1]
                    sig.append((k4 / len(a4) < k23 / len(a23)) and p < 0.05)
                    rec[f"{'BW' if meas == 'bw' else 'eBW'} {b} L4/L23 %"] = f"{100 * k4 / len(a4):.0f}/{100 * k23 / len(a23):.0f} p={p:.3g}"
                rec[f"{'BW' if meas == 'bw' else 'eBW'} bins L4 lower"] = int(sum(sig))
            rows.append(rec)
l4t = pd.DataFrame(rows)
print(l4t.T.to_string())

**Result: not reproduced.** In NH L4 (C1-C3, Methods' test, E1) 63.2% of the 57 entries within 75 um
have the CW as BW and 73.7% among the eBWs (paper 63% and 73%, both within tolerance), and the BW
fraction does not differ from L2/3 (54.2%, Fisher p = 0.43). But the eBW fraction is lower in L4 than
in L2/3 (73.7% against 91.7%, p = 0.022), against the claim's absence, because the L2/3 eBW fraction
is far above the paper's. By eBW, L4 is lower than L2/3 in all three bins from 150 to 375 um (p from
2e-11 to 8e-9). Under E3 the 0-75 um eBW test is not significant (p = 0.28), but the L4 value is
80.7%, outside tolerance.

### fig3a-l23-enrichment-more-cw-tuned

> In L2/3, EN raised the fraction of cells tuned to the CW at column centres (82%, 111/135, against
> 67% in NH) and in every bin below 300 um, not beyond; over all cells within column boundaries
> 77.1% (468/607) against 65.1% (224/344).

In [ ]:
def within_columns(tb, ar, layer, sel, ek):
    out = {}
    for g in ("NH", "EN"):
        m = (tb.layer == layer).values & (tb.group == g).values & tb.inb.values
        if sel is not None:
            m = m & sel
        idx = np.where(m)[0]
        cw = np.array([ssc10.WHISKERS.index(b) for b in tb.barrel.values[idx]])
        out[g] = ar[ek][idx, cw]
    k = [int(out[g].sum()) for g in ("NH", "EN")]
    n = [len(out[g]) for g in ("NH", "EN")]
    p = stats.fisher_exact([[k[1], n[1] - k[1]], [k[0], n[0] - k[0]]])[1]
    return k, n, p


rows, bins_tab = [], {}
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for rr in ("A", "B"):
        e = ENT[("L23", resp, rr)]
        for ek in ("e1", "e2", "e3"):
            if ek == "e3" and resp == "stored":
                continue
            per = []
            for b in [f"{BINS[i]}-{BINS[i + 1]}" for i in range(len(BINS) - 1)]:
                s = e[e.bin == b]
                kn, nn = int(s[s.group == "NH"][ek].sum()), int((s.group == "NH").sum())
                ke, ne = int(s[s.group == "EN"][ek].sum()), int((s.group == "EN").sum())
                p = stats.fisher_exact([[ke, ne - ke], [kn, nn - kn]])[1]
                p_perm = ssc10.perm_test_means(s[s.group == "EN"][ek].astype(float), s[s.group == "NH"][ek].astype(float),
                                               np.random.default_rng(402))[1]
                per.append(dict(bin=b, NH=f"{kn}/{nn}", EN=f"{ke}/{ne}", NH_pct=100 * kn / nn, EN_pct=100 * ke / ne, p=p, p_perm=p_perm,
                                en_higher_sig=(ke / ne > kn / nn) and p < 0.05))
            per = pd.DataFrame(per).set_index("bin")
            bins_tab[(resp, rr, ek)] = per
            kc, nc, pc = within_columns(tb, ar, "L23", None, ek)
            near = per.loc[["0-75", "75-150", "150-225", "225-300"]]
            far = per.drop(["0-75", "75-150", "150-225", "225-300"])
            rows.append(dict(responsive=resp, rois=rr, ebw=ek,
                             EN_0_75=round(per.loc["0-75", "EN_pct"], 1), NH_0_75=round(per.loc["0-75", "NH_pct"], 1),
                             p_0_75=round(per.loc["0-75", "p"], 4),
                             EN_col=f"{kc[1]}/{nc[1]} = {100 * kc[1] / nc[1]:.1f}", NH_col=f"{kc[0]}/{nc[0]} = {100 * kc[0] / nc[0]:.1f}",
                             p_col=f"{pc:.2g}", near_bins_sig=int(near.en_higher_sig.sum()), far_bins_sig=int(far.en_higher_sig.sum())))
f3a = pd.DataFrame(rows)
print(f3a.to_string(index=False))
print("\nMethods responsiveness, reading A, E1, per bin:")
print(bins_tab[("methods", "A", "e1")].round(4).to_string())

**Result: not reproduced.** At 0-75 um the EN fraction tuned to the CW is lower than NH under every
reading (for example 82.6% against 91.7%, Fisher p = 0.16, Methods' test, E1), and over ROIs within
columns the fractions are equal (84.7% of 635 against 84.8% of 376, p = 1). EN is not significantly
higher in any bin below 300 um; it is significantly lower from 375 to 675 um. The EN values (82.6%,
84.7%) are near the paper's (82%, 77.1%); the NH values (91.7%, 84.8%) are far above its 67% and
65.1%.

### fig3c-l4-enrichment-more-cw-tuned

> In L4, EN also raised the fraction of CW-tuned cells, at column centres (90%, 57/63, against 73%)
> and over all cells within columns (89.3%, 133/149, against 75.6%, 65/86).

In [ ]:
rows = []
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for rr, sel in (("C1-C3", tb.c123.values), ("all L4", None)):
        e = ENT[("L4", resp, rr)]
        for ek in ("e1", "e2", "e3"):
            if ek == "e3" and resp == "stored":
                continue
            s = e[e.bin == "0-75"]
            kn, nn = int(s[s.group == "NH"][ek].sum()), int((s.group == "NH").sum())
            ke, ne = int(s[s.group == "EN"][ek].sum()), int((s.group == "EN").sum())
            p = stats.fisher_exact([[ke, ne - ke], [kn, nn - kn]])[1]
            kc, nc, pc = within_columns(tb, ar, "L4", sel, ek)
            rows.append(dict(responsive=resp, rois=rr, ebw=ek, EN_0_75=f"{ke}/{ne} = {100 * ke / ne:.1f}",
                             NH_0_75=f"{kn}/{nn} = {100 * kn / nn:.1f}", p_0_75=round(p, 4),
                             EN_col=f"{kc[1]}/{nc[1]} = {100 * kc[1] / nc[1]:.1f}", NH_col=f"{kc[0]}/{nc[0]} = {100 * kc[0] / nc[0]:.1f}",
                             p_col=round(pc, 4)))
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** In L4 C1-C3 (Methods' test, E1), 97.0% of EN entries at 0-75 um (65/67) are
tuned to the CW against 73.7% in NH (42/57), Fisher p = 0.0002; within columns 93.2% (151/162)
against 76.8% (73/95), p = 0.0004. All four values are within tolerance (paper 90, 73, 89.3, 75.6).
Every other reading gives EN higher with p < 0.005.

## Receptive fields (Figure 4)

Responses are the z-scored medians (`z`; `zmed` as the variant). EN against NH by the two-sided
permutation test of the difference in means, ROIs as units.

In [ ]:
def ranked(tb, ar, layer, sel=None, zkey="z"):
    m = (tb.layer == layer).values
    if sel is not None:
        m = m & sel
    out = {}
    for g in ("NH", "EN"):
        z = ar[zkey][m & (tb.group == g).values]
        z = z[np.isfinite(z).all(1)]           # ROIs with all nine whiskers (DNH13b lacks the E row)
        out[g] = -np.sort(-z, axis=1)
    return out


def rank_tests(rk, rng):
    rows = []
    for k in range(9):
        d, p = ssc10.perm_test_means(rk["EN"][:, k], rk["NH"][:, k], rng)
        rows.append(dict(rank=k + 1, NH=rk["NH"][:, k].mean(), EN=rk["EN"][:, k].mean(), diff=d, p=p))
    return pd.DataFrame(rows).set_index("rank")


RANKED = {}
rng = np.random.default_rng(501)
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for rr, sel in (("A", None), ("B", tb.inb.values)):
        for zkey in ("z", "zmed"):
            RANKED[(resp, rr, zkey)] = rank_tests(ranked(tb, ar, "L23", sel, zkey), rng)
print("n ROIs with all nine whiskers (Methods, A):", {g: len(v) for g, v in ranked(T, A, "L23").items()})

### fig4a-l23-ranked-rf-sharper

> In L2/3, EN increased responses to the strongest three whiskers and decreased responses to the
> weakest three in rank-ordered receptive fields normalised to spontaneous activity, so receptive
> fields were sharper.

In [ ]:
rows = []
for key, rt in RANKED.items():
    rec = dict(responsive=key[0], rois=key[1], zscore=key[2])
    for k in range(1, 10):
        rec[f"r{k}"] = f"{rt.loc[k, 'diff']:+.3f} ({rt.loc[k, 'p']:.3g})"
    rec["top3 EN higher p<0.05"] = int(sum((rt.loc[k, "diff"] > 0) and rt.loc[k, "p"] < 0.05 for k in (1, 2, 3)))
    rec["bottom3 EN lower p<0.05"] = int(sum((rt.loc[k, "diff"] < 0) and rt.loc[k, "p"] < 0.05 for k in (7, 8, 9)))
    rec["direction at six ranks"] = int(sum(rt.loc[k, "diff"] > 0 for k in (1, 2, 3)) + sum(rt.loc[k, "diff"] < 0 for k in (7, 8, 9)))
    rows.append(rec)
print("EN - NH (p) per rank:")
print(pd.DataFrame(rows).to_string(index=False))
rt = RANKED[("methods", "A", "z")]
print("\nMethods, A, z:")
print(rt.round(4).to_string())

**Result: reproduced.** Under the Methods' test, reading A, blank-mean z, EN is higher at ranks 1 to 3
(+0.078, +0.075, +0.032; p = 0.0018, 0.0002, 0.026) and lower at ranks 7 to 9 (-0.035, -0.034,
-0.045; p = 0.0006, 0.0008, 0.0002). Every reading has the direction at all six ranks; the weakest
ranks are significant in all three only under the blank-mean z.

### fig4b-l23-cw-and-best-sw-stronger

> In L2/3, EN increased the signal-to-noise of the CW response and of the two best surround whisker
> responses.

In [ ]:
def cw_sw(tb, ar, layer, sel=None, zkey="z"):
    m = (tb.layer == layer).values & tb.inb.values
    if sel is not None:
        m = m & sel
    out = {}
    for g in ("NH", "EN"):
        idx = np.where(m & (tb.group == g).values)[0]
        cw = np.array([ssc10.WHISKERS.index(b) for b in tb.barrel.values[idx]])
        z = ar[zkey][idx]
        cwz = z[np.arange(len(idx)), cw]
        sw = z.copy(); sw[np.arange(len(idx)), cw] = np.nan
        sw = -np.sort(-np.where(np.isfinite(sw), sw, -np.inf), axis=1)[:, :8]
        sw[~np.isfinite(sw)] = np.nan
        out[g] = (cwz, sw)
    return out


rng = np.random.default_rng(502)
rows = []
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for zkey in ("z", "zmed"):
        o = cw_sw(tb, ar, "L23", None, zkey)
        rec = dict(responsive=resp, zscore=zkey, n=(len(o["NH"][0]), len(o["EN"][0])))
        d, p = ssc10.perm_test_means(o["EN"][0], o["NH"][0], rng)
        rec["CW NH/EN"] = f"{o['NH'][0].mean():.3f}/{o['EN'][0].mean():.3f}"; rec["CW p"] = p
        ok = [d > 0 and p < 0.05]
        for k in range(3):
            d, p = ssc10.perm_test_means(o["EN"][1][:, k], o["NH"][1][:, k], rng)
            rec[f"SW{k + 1} NH/EN"] = f"{np.nanmean(o['NH'][1][:, k]):.3f}/{np.nanmean(o['EN'][1][:, k]):.3f}"; rec[f"SW{k + 1} p"] = p
            if k < 2:
                ok.append(d > 0 and p < 0.05)
        rec["CW, SW1, SW2 EN higher p<0.05"] = int(sum(ok))
        rows.append(rec)
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** With the blank-median z (Methods' test) the CW response is 0.642 in NH and
0.703 in EN (p = 0.048), the best SW 0.618 and 0.717 (p = 0.0001) and the second 0.434 and 0.488
(p = 0.0045); the stored p-values give the same under both z-scores (CW p = 0.011 and 0.015). With
the blank-mean z and the Methods' test the CW difference is not significant (p = 0.14).

### fig4c-spontaneous-activity

> EN decreased spontaneous (blank trial) activity in L2/3, while L4 spontaneous activity was
> unaffected.

Spontaneous activity: the median over blank trials of the evoked-window response (baseline
subtracted); the variant is the median evoked-window dF/F without baseline subtraction
(`spont_level`, from the stored `deltaF` at the blank onsets of `Stimuli`).

In [ ]:
rng = np.random.default_rng(503)
rows = []
for resp, tb in (("methods", T), ("stored", TS)):
    for key in ("spont", "spont_level"):
        for lay, sel in (("L23", None), ("L4", tb.c123.values), ("L4 all", None)):
            m = (tb.layer == lay.split()[0]).values
            if sel is not None:
                m = m & sel
            a, b = tb[m & (tb.group == "EN").values][key], tb[m & (tb.group == "NH").values][key]
            d, p = ssc10.perm_test_means(a, b, rng)
            rows.append(dict(responsive=resp, measure=key, layer=lay, NH=b.mean(), EN=a.mean(), diff=d, p=p, n=(len(b), len(a))))
print(pd.DataFrame(rows).round(5).to_string(index=False))

**Result: reproduced, under the claims file's variant.** Measured as the median dF/F in the
evoked window of blank trials without baseline subtraction, spontaneous activity in L2/3 is lower in
EN (0.0834 against 0.0914, p = 0.0001) and does not differ in L4 C1-C3 (0.198 against 0.207,
p = 0.20); the stored p-values give the same (p = 0.0003 and 0.64). With the baseline-subtracted
measure, the claims file's main reading, L2/3 spontaneous activity is higher in EN (-0.0027 against
-0.0064, p = 0.0001), the opposite direction, and L4 shows no difference (p = 0.51). Over all L4
ROIs, not only C1-C3, EN is lower by the variant (p = 0.0001).

### figs4-1-absolute-dff

> Without normalisation to spontaneous activity, L2/3 cells in EN had greater mean evoked dF/F than
> NH, and the regression of weakest-four on strongest-four whisker responses had a different slope in
> EN, showing sharper tuning.

Per ROI the mean evoked dF/F per whisker (mean over trials), sorted; the four strongest and four
weakest are averaged. Slopes are compared by the interaction term of an OLS fit.

In [ ]:
rng = np.random.default_rng(504)
rows = []
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    m = (tb.layer == "L23").values
    d = []
    for g in ("NH", "EN"):
        mm = ar["mean"][m & (tb.group == g).values]
        mm = mm[np.isfinite(mm).all(1)]
        s = -np.sort(-mm, axis=1)
        d.append(pd.DataFrame(dict(group=g, strong=s[:, :4].mean(1), weak=s[:, 5:].mean(1), all9=mm.mean(1))))
    d = pd.concat(d)
    rec = dict(responsive=resp, n=(int((d.group == "NH").sum()), int((d.group == "EN").sum())))
    for k in ("strong", "all9"):
        diff, p = ssc10.perm_test_means(d[d.group == "EN"][k], d[d.group == "NH"][k], rng)
        rec[f"{k} NH/EN"] = f"{d[d.group == 'NH'][k].mean():.4f}/{d[d.group == 'EN'][k].mean():.4f}"; rec[f"{k} p"] = p
    fit = smf.ols("weak ~ strong * C(group, Treatment('NH'))", data=d).fit()
    term = [t for t in fit.params.index if t.startswith("strong:")][0]
    for g in ("NH", "EN"):
        rec[f"slope {g}"] = round(smf.ols("weak ~ strong", data=d[d.group == g]).fit().params["strong"], 4)
    rec["slope diff (EN-NH)"] = round(fit.params[term], 4); rec["t"] = round(fit.tvalues[term], 2); rec["p slope"] = fit.pvalues[term]
    rows.append(rec)
print(pd.DataFrame(rows).T.to_string())

**Result: not reproduced.** The mean dF/F of the four strongest whiskers does not differ (NH 0.0275,
EN 0.0280, p = 0.65); the mean over all nine is higher in EN (0.0147 against 0.0130, p = 0.010). The
regression of the weakest four on the strongest four has a higher slope in EN (0.123 against
-0.019; difference t = 8.3, p < 1e-15), the opposite of the sharper tuning the claim reads from it.

### figs4-2-barrel-vs-septal

> Sharpening of L2/3 tuning by EN was more prominent for ROIs over L4 barrels than for ROIs over septa.

In [ ]:
rng = np.random.default_rng(505)
rows = []
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for zkey in ("z", "zmed"):
        rec = dict(responsive=resp, zscore=zkey)
        for name, sel in (("barrel", tb.inb.values), ("septal", ~tb.inb.values)):
            rk = ranked(tb, ar, "L23", sel, zkey)
            rt = rank_tests(rk, rng)
            rec[f"{name} n"] = (len(rk["NH"]), len(rk["EN"]))
            rec[f"{name} sig ranks"] = int((rt.p < 0.05).sum())
            rec[f"{name} top3 diff"] = round(rt.loc[[1, 2, 3], "diff"].mean(), 3)
            rec[f"{name} bottom3 diff"] = round(rt.loc[[7, 8, 9], "diff"].mean(), 3)
            rec[f"{name} per rank"] = " ".join(f"{x:+.2f}{'*' if q < 0.05 else ''}" for x, q in zip(rt["diff"], rt.p))
        rows.append(rec)
print(pd.DataFrame(rows).T.to_string())

**Result: reproduced.** EN differs significantly from NH at more ranks for ROIs over barrels than over
septa under every reading (6 against 4, 4 against 1, 6 against 2, 5 against 2), and the top-three
increase is larger over barrels (+0.080 to +0.117 against +0.004 to +0.070). The septal groups are
smaller (90 NH and 210 EN ROIs against 346 and 635), which alone lowers their count, and under the
Methods' test with the blank-mean z the septal group has the larger decrease at the weakest ranks
(-0.068 against -0.030).

### Somatotopic receptive fields (Figure 4D and 4E)

For ROIs over a barrel, the response to the CW and to its neighbours: -1 row is the whisker one row
dorsal in the same arc (C2 for D2), +1 row one row ventral (E2), -1 arc one arc caudal in the same
row (D1), +1 arc one arc rostral (D3), for ROIs whose CW has that neighbour among the nine.
"Columns with at least 3 ROIs" is applied per field and barrel; all columns is the other reading.

In [ ]:
def neighbour(cw, dr, da):
    r, a = "cde".index(cw[0]) + dr, int(cw[1]) + da
    return f"{'cde'[r]}{a}" if 0 <= r < 3 and 1 <= a <= 3 else None


def somato(tb, ar, layer, sel=None, zkey="z", min_col=None):
    m = (tb.layer == layer).values & tb.inb.values
    if sel is not None:
        m = m & sel
    if min_col:
        cnt = tb[m].groupby(["field", "barrel"]).size()
        okc = set(cnt[cnt >= min_col].index)
        m = m & np.array([(f, b) in okc for f, b in zip(tb.field, tb.barrel)])
    out = {}
    for g in ("NH", "EN"):
        idx = np.where(m & (tb.group == g).values)[0]
        rec = {"CW": [], "-1 row": [], "+1 row": [], "-1 arc": [], "+1 arc": []}
        for i in idx:
            cw = tb.barrel.iat[i]
            rec["CW"].append(ar[zkey][i, ssc10.WHISKERS.index(cw)])
            for name, dr, da in (("-1 row", -1, 0), ("+1 row", 1, 0), ("-1 arc", 0, -1), ("+1 arc", 0, 1)):
                nb = neighbour(cw, dr, da)
                if nb is not None:
                    rec[name].append(ar[zkey][i, ssc10.WHISKERS.index(nb)])
        out[g] = {k: np.array(v, float) for k, v in rec.items()}
    return out


def somato_table(o, rng):
    rows = []
    for k in ("CW", "-1 row", "+1 row", "-1 arc", "+1 arc"):
        a, b = o["EN"][k], o["NH"][k]
        d, p = ssc10.perm_test_means(a, b, rng)
        rows.append(dict(whisker=k, n_NH=int(np.isfinite(b).sum()), NH=np.nanmean(b), sem_NH=ssc10.sem(b),
                         n_EN=int(np.isfinite(a).sum()), EN=np.nanmean(a), sem_EN=ssc10.sem(a), diff=d, p=p))
    return pd.DataFrame(rows).set_index("whisker")


SOM = {}
rng = np.random.default_rng(506)
for resp, tb, ar in (("methods", T, A), ("stored", TS, AS)):
    for mc in (3, None):
        for zkey in ("z", "zmed"):
            SOM[("L23", resp, mc, zkey)] = somato_table(somato(tb, ar, "L23", None, zkey, mc), rng)
            SOM[("L4", resp, mc, zkey)] = somato_table(somato(tb, ar, "L4", tb.c123.values, zkey, mc), rng)
print("L2/3, Methods responsiveness, columns with >= 3 ROIs, z:")
print(SOM[("L23", "methods", 3, "z")].round(4).to_string())
print("\nL4 (C1-C3), Methods responsiveness, columns with >= 3 ROIs, z:")
print(SOM[("L4", "methods", 3, "z")].round(4).to_string())

### fig4d-l23-cw-stronger

> In L2/3, CW responses were higher in EN than NH (0.71 against 0.64).

In [ ]:
rows = []
for (lay, resp, mc, zkey), s in SOM.items():
    if lay != "L23":
        continue
    rows.append(dict(responsive=resp, min_rois_per_column=mc, zscore=zkey, NH=round(s.loc["CW", "NH"], 3), EN=round(s.loc["CW", "EN"], 3),
                     p=s.loc["CW", "p"], n=(s.loc["CW", "n_NH"], s.loc["CW", "n_EN"]),
                     within=(abs(s.loc["CW", "NH"] - 0.64) <= 0.03) and (abs(s.loc["CW", "EN"] - 0.71) <= 0.02)))
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced, under one reading.** With the stored p-values, columns of at least 3 ROIs and
the blank-mean z, CW responses are 0.620 in NH and 0.699 in EN (p = 0.015), both within tolerance;
all four stored-p readings give p from 0.010 to 0.018. With the Methods' test the values are also
within tolerance (0.661 and 0.704) but the difference is not significant (p = 0.05 to 0.16).

### fig4d-l23-across-row-sharpening

> In L2/3, EN reduced responses to the surround whiskers one row dorsal and one row ventral to the
> CW (from 68% and 62% of the CW response in NH to 49% and 41% in EN), sharpening receptive fields
> across rows.

In [ ]:
rows = []
for (lay, resp, mc, zkey), s in SOM.items():
    if lay != "L23":
        continue
    rec = dict(responsive=resp, min_rois_per_column=mc, zscore=zkey)
    within = 0
    for k, (nh, tnh, en, ten, fn, fe) in {"-1 row": (0.44, 0.02, 0.35, 0.02, 68, 49), "+1 row": (0.39, 0.02, 0.29, 0.01, 62, 41)}.items():
        rec[f"{k} NH/EN"] = f"{s.loc[k, 'NH']:.3f}/{s.loc[k, 'EN']:.3f}"; rec[f"{k} p"] = s.loc[k, "p"]
        pn, pe = 100 * s.loc[k, "NH"] / s.loc["CW", "NH"], 100 * s.loc[k, "EN"] / s.loc["CW", "EN"]
        rec[f"{k} % of CW NH/EN"] = f"{pn:.0f}/{pe:.0f}"
        within += (abs(s.loc[k, "NH"] - nh) <= tnh) + (abs(s.loc[k, "EN"] - en) <= ten)
        rec[f"{k} EN lower p<0.05"] = (s.loc[k, "diff"] < 0) and s.loc[k, "p"] < 0.05
    rec["responses within (of 4)"] = int(within)
    rows.append(rec)
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** EN lowers the -1 row response (0.404 to 0.334, p = 0.022) and the
+1 row response (0.394 to 0.278, p = 0.0001) (Methods' test, columns of at least 3 ROIs, blank-mean
z), to 47% and 40% of the CW response from 61% and 60% (paper 68% and 62% to 49% and 41%). Two of
the four response values are within tolerance (NH +1 row 0.394, EN -1 row 0.334); NH -1 row (0.404
against 0.44 +/- 0.02) and EN +1 row (0.278 against 0.29 +/- 0.01) are not. With the stored p-values the -1 row difference is not significant
(p = 0.083 to 0.098). Only 157 NH ROIs have a -1 row neighbour, as NH ROIs over a barrel lie mostly
in the C and D rows.

### fig4d-l23-within-row-broadening

> In the within-row dimension, EN increased L2/3 responses to the surround whisker one arc rostral of
> the CW (0.34 to 0.53) and did not alter responses to the one arc caudal (0.57 and 0.55),
> broadening receptive fields along rows.

In [ ]:
rows = []
for (lay, resp, mc, zkey), s in SOM.items():
    if lay != "L23":
        continue
    rows.append(dict(responsive=resp, min_rois_per_column=mc, zscore=zkey,
                     plus_arc=f"{s.loc['+1 arc', 'NH']:.3f}/{s.loc['+1 arc', 'EN']:.3f}", plus_p=s.loc["+1 arc", "p"],
                     minus_arc=f"{s.loc['-1 arc', 'NH']:.3f}/{s.loc['-1 arc', 'EN']:.3f}", minus_p=s.loc["-1 arc", "p"],
                     n_plus=(s.loc["+1 arc", "n_NH"], s.loc["+1 arc", "n_EN"]), n_minus=(s.loc["-1 arc", "n_NH"], s.loc["-1 arc", "n_EN"]),
                     within=int(abs(s.loc["+1 arc", "NH"] - 0.34) <= 0.01) + int(abs(s.loc["+1 arc", "EN"] - 0.53) <= 0.02)
                     + int(abs(s.loc["-1 arc", "NH"] - 0.57) <= 0.04) + int(abs(s.loc["-1 arc", "EN"] - 0.55) <= 0.04)))
print(pd.DataFrame(rows).to_string(index=False))
# Where the ROIs over a barrel lie, by group: which CWs have a +1 or -1 arc neighbour
print("\nL2/3 responsive ROIs over a barrel, by barrel and group (Methods):")
print(T[(T.layer == "L23") & T.inb].groupby(["group", "barrel"]).size().unstack(0).to_string())

**Result: partially reproduced.** EN raises the +1 arc response (0.339 to 0.525 with the stored
p-values and blank-mean z; p = 0.0001 under every reading) and does not change the -1 arc response
(0.460 and 0.530, p = 0.17; p = 0.14 to 0.46 over the readings). Three of four values are within
tolerance in that reading; the NH -1 arc response is 0.460 against 0.57 +/- 0.04, and is 0.50 or
less in every reading.

### fig4ae-l4-weaker-responses

> In L4, EN reduced responses to most whiskers in ranked receptive fields and, in somatotopic space,
> to the CW (0.98 to 0.81) and to the +1 arc (0.54 to 0.17), -1 arc (0.67 to 0.33) and +1 row (0.23
> to 0.11) surround whiskers, sharpening receptive fields by global weakening.

In [ ]:
EXP4 = {"CW": (0.98, 0.09, 0.81, 0.04), "+1 arc": (0.54, 0.08, 0.17, 0.02), "-1 arc": (0.67, 0.13, 0.33, 0.04), "+1 row": (0.23, 0.03, 0.11, 0.02)}
rng = np.random.default_rng(507)
rows = []
for (lay, resp, mc, zkey), s in SOM.items():
    if lay != "L4":
        continue
    tb, ar = (T, A) if resp == "methods" else (TS, AS)
    rt = rank_tests(ranked(tb, ar, "L4", tb.c123.values, zkey), rng)
    rec = dict(responsive=resp, min_rois_per_column=mc, zscore=zkey)
    tests, within = 0, 0
    for k, (nh, tnh, en, ten) in EXP4.items():
        rec[k] = f"{s.loc[k, 'NH']:.2f}/{s.loc[k, 'EN']:.2f} p={s.loc[k, 'p']:.3g} n={s.loc[k, 'n_NH']}/{s.loc[k, 'n_EN']}"
        tests += (s.loc[k, "diff"] < 0) and s.loc[k, "p"] < 0.05
        within += (abs(s.loc[k, "NH"] - nh) <= tnh) + (abs(s.loc[k, "EN"] - en) <= ten)
    rec["tests EN lower (of 4)"] = int(tests); rec["values within (of 8)"] = int(within)
    rec["ranks EN lower p<0.05"] = int(((rt["diff"] < 0) & (rt.p < 0.05)).sum())
    rows.append(rec)
print(pd.DataFrame(rows).T.to_string())

**Result: not reproduced.** In L4 (C1-C3) EN lowers the +1 arc (0.53 to 0.19, p = 0.0001), -1 arc
(0.66 to 0.31, p = 0.0014) and +1 row (0.24 to 0.14, p = 0.0022) responses, and 8 of 9 ranks of the
ranked receptive field (Methods' test, columns of at least 3 ROIs, blank-mean z), but the CW
response is not significantly lower (0.93 against 0.86, p = 0.39; p = 0.18 to 0.87 over the
readings). Four of the eight values are within tolerance.

## Correlations (Figures 5 and 6)

Pairs of simultaneously imaged responsive ROIs that both lie over (L2/3) or in (L4) a barrel
(`ROIsInBarrel`); within-column when the barrel is the same. Tests are two-sided permutation tests
of the difference in means with pairs as units.

In [ ]:
t0 = time.time()
PAIRS = {"methods": ssc10.pair_table(F, "methods"), "stored": ssc10.pair_table(F, "stored")}
print(f"pair tables: {time.time() - t0:.0f} s")
for k, p in PAIRS.items():
    print(k, p.groupby(["layer", "group", "same"]).size().unstack().to_string(), "\n")
pp = PAIRS["methods"]
print("Signal correlation range", pp.signal.min().round(3), "to", pp.signal.max().round(3), "; noise", pp.noise.min().round(3), "to", pp.noise.max().round(3),
      "; pair distance", pp.dist.min().round(1), "to", pp.dist.max().round(1), "um")

### fig5a-l23-signal-correlation-boundary

> In L2/3, EN modestly raised signal correlation for within-column pairs (0.637 to 0.659) and greatly
> reduced it for across-column pairs (0.541 to 0.425); in EN, closely spaced pairs (under 150 um) had
> much lower signal correlation across than within columns (0.529 against 0.674), where in NH the
> difference was modest (0.611 against 0.652).

In [ ]:
def corr_summary(pp, layer, key, rng):
    p = pp[pp.layer == layer]
    out = {}
    for g in ("NH", "EN"):
        for same in (True, False):
            s = p[(p.group == g) & (p.same == same)][key]
            out[(g, same, "all")] = s
            out[(g, same, "<150")] = p[(p.group == g) & (p.same == same) & (p.dist < 150)][key]
    res = {}
    for k, s in out.items():
        res[k] = (s.mean(), ssc10.sem(s), len(s))
    tests = {
        "within EN vs NH": ssc10.perm_test_means(out[("EN", True, "all")], out[("NH", True, "all")], rng),
        "across EN vs NH": ssc10.perm_test_means(out[("EN", False, "all")], out[("NH", False, "all")], rng),
        "NH <150 within vs across": ssc10.perm_test_means(out[("NH", True, "<150")], out[("NH", False, "<150")], rng),
        "EN <150 within vs across": ssc10.perm_test_means(out[("EN", True, "<150")], out[("EN", False, "<150")], rng),
    }
    return res, tests


rng = np.random.default_rng(601)
EXP5A = {("NH", True, "<150"): (0.652, 0.005), ("NH", False, "<150"): (0.611, 0.007), ("NH", True, "all"): (0.637, 0.004),
         ("EN", True, "all"): (0.659, 0.002), ("NH", False, "all"): (0.541, 0.004), ("EN", False, "all"): (0.425, 0.004),
         ("EN", True, "<150"): (0.674, 0.003), ("EN", False, "<150"): (0.529, 0.006)}
for resp, pp in PAIRS.items():
    res, tests = corr_summary(pp, "L23", "signal", rng)
    print(f"Responsiveness: {resp}")
    tab = pd.DataFrame([dict(group=k[0], within=k[1], pairs=k[2], mean=v[0], sem=v[1], n=v[2],
                             paper=EXP5A[k][0], within_tol=abs(v[0] - EXP5A[k][0]) <= EXP5A[k][1]) for k, v in res.items()])
    print(tab.round(4).to_string(index=False))
    for k, (d, p) in tests.items():
        print(f"  {k}: difference {d:+.4f}, p = {p:.4g}")
    print()

In [ ]:
# By field: mean within- and across-column signal correlation (Methods responsiveness)
pp = PAIRS["methods"]
byf = pp[pp.layer == "L23"].groupby(["group", "field", "same"]).signal.agg(["mean", "size"]).unstack("same").round(3)
print(byf.to_string())

**Result: partially reproduced.** All four tests hold under both responsiveness readings: EN raises
within-column signal correlation (0.586 to 0.599, p = 0.012, Methods' test) and lowers it across
columns (0.467 to 0.388, p = 0.0001), and among pairs under 150 um the within minus across
difference is +0.059 in NH and +0.143 in EN (both p = 0.0001). Every value is 0.04 to 0.08 below the
paper's, outside its tolerances.

### fig5b-l23-noise-correlation-boundary

> In L2/3 of NH mice, noise correlations were similar within (0.334) and across (0.327) columns; EN
> modestly reduced them within columns (0.307) and greatly reduced them across columns (0.255), even
> for pairs about 100 um apart.

In [ ]:
rng = np.random.default_rng(602)
EXP5B = {("NH", True): (0.334, 0.002), ("NH", False): (0.327, 0.002), ("EN", True): (0.307, 0.001), ("EN", False): (0.255, 0.001)}
for resp, pp in PAIRS.items():
    p = pp[pp.layer == "L23"]
    print(f"Responsiveness: {resp}")
    for (g, same), (e, tol) in EXP5B.items():
        s = p[(p.group == g) & (p.same == same)].noise
        print(f"  {g} {'within' if same else 'across'}: {s.mean():.4f} +/- {ssc10.sem(s):.4f} (n = {len(s)}); paper {e} +/- {tol}; within: {abs(s.mean() - e) <= tol}")
    w = ssc10.perm_test_means(p[(p.group == "EN") & p.same].noise, p[(p.group == "NH") & p.same].noise, rng)
    a = ssc10.perm_test_means(p[(p.group == "EN") & ~p.same].noise, p[(p.group == "NH") & ~p.same].noise, rng)
    near = p[~p.same & (p.dist >= 75) & (p.dist < 125)]
    n = ssc10.perm_test_means(near[near.group == "EN"].noise, near[near.group == "NH"].noise, rng)
    nhw = ssc10.perm_test_means(p[(p.group == "NH") & p.same].noise, p[(p.group == "NH") & ~p.same].noise, rng)
    print(f"  within EN - NH {w[0]:+.4f} (p = {w[1]:.4g}); across EN - NH {a[0]:+.4f} (p = {a[1]:.4g}); across reduction larger: {a[0] < w[0]}")
    print(f"  across-column pairs 75-125 um apart: NH {near[near.group == 'NH'].noise.mean():.4f} (n = {(near.group == 'NH').sum()}),"
          f" EN {near[near.group == 'EN'].noise.mean():.4f} (n = {(near.group == 'EN').sum()}), p = {n[1]:.4g}")
    print(f"  NH within - across {nhw[0]:+.4f} (p = {nhw[1]:.4g}; the paper reports no test)\n")

**Result: partially reproduced.** EN lowers noise correlation within columns (Methods' test: 0.344 to
0.315, p = 0.0001) and more across columns (0.319 to 0.265, p = 0.0001), and across-column pairs 75
to 125 um apart (0.305 to 0.290, p = 0.0091). With the stored p-values the values are 0.332, 0.320,
0.310 and 0.257, of which only the first is within the tight tolerance (+/- 0.001 to 0.002).

### fig5eh-l23-correlation-gradient-at-edge

> EN substantially reduced the signal and noise correlations of neurons outside a reference column to
> neurons inside it, while preserving signal correlation and modestly reducing noise correlation
> within the column, giving a steeper gradient at the column edge.

For each responsive L2/3 ROI over a barrel and each reference column of its field holding other
such ROIs, the mean correlation to the ROIs in that column; inside when the ROI is in the column.

In [ ]:
def to_column(pp, layer="L23"):
    p = pp[pp.layer == layer]
    both = pd.concat([p.rename(columns={"i": "a", "j": "b", "bi": "ba", "bj": "bb"}),
                      p.rename(columns={"j": "a", "i": "b", "bj": "ba", "bi": "bb"})])
    g = both.groupby(["group", "field", "a", "ba", "bb"])[["signal", "noise"]].mean().reset_index()
    g["inside"] = g.ba == g.bb
    return g


rng = np.random.default_rng(603)
for resp, pp in PAIRS.items():
    g = to_column(pp)
    print(f"Responsiveness: {resp}; ROI-column entries inside {int(g.inside.sum())}, outside {int((~g.inside).sum())}")
    for key in ("signal", "noise"):
        ins = ssc10.perm_test_means(g[(g.group == "EN") & g.inside][key], g[(g.group == "NH") & g.inside][key], rng)
        out = ssc10.perm_test_means(g[(g.group == "EN") & ~g.inside][key], g[(g.group == "NH") & ~g.inside][key], rng)
        print(f"  {key}: inside NH {g[(g.group == 'NH') & g.inside][key].mean():.4f} EN {g[(g.group == 'EN') & g.inside][key].mean():.4f}"
              f" (EN - NH {ins[0]:+.4f}, p = {ins[1]:.4g}); outside NH {g[(g.group == 'NH') & ~g.inside][key].mean():.4f}"
              f" EN {g[(g.group == 'EN') & ~g.inside][key].mean():.4f} (EN - NH {out[0]:+.4f}, p = {out[1]:.4g});"
              f" more negative outside: {out[0] < ins[0]}")
    print()

**Result: reproduced, under one reading.** With the stored p-values, EN lowers the correlations of
ROIs outside a column to that column (signal -0.043, p = 0.0009; noise -0.043, p = 0.0001), more
than inside for both (inside +0.018 and -0.041), and inside the column the signal correlation is not
lower (p = 0.19). With the Methods' test all of this holds except that the noise reduction is the
same outside and inside (-0.0459 against -0.0461).

### figs5-1-row-and-arc-boundaries

> The reduction of signal and noise correlations across column boundaries appeared for both
> within-row and within-arc column boundaries.

In [ ]:
rng = np.random.default_rng(604)
for resp, pp in PAIRS.items():
    p = pp[(pp.layer == "L23") & ~pp.same].copy()
    p["kind"] = np.where(p.bi.str[0] == p.bj.str[0], "same row", np.where(p.bi.str[1] == p.bj.str[1], "same arc", "diagonal"))
    print(f"Responsiveness: {resp}")
    for kind in ("same row", "same arc", "diagonal"):
        s = p[p.kind == kind]
        for key in ("signal", "noise"):
            d, pv = ssc10.perm_test_means(s[s.group == "EN"][key], s[s.group == "NH"][key], rng)
            print(f"  {kind:8s} {key:6s}: NH {s[s.group == 'NH'][key].mean():.4f} (n = {(s.group == 'NH').sum()}),"
                  f" EN {s[s.group == 'EN'][key].mean():.4f} (n = {(s.group == 'EN').sum()}), EN - NH {d:+.4f}, p = {pv:.4g}")

**Result: not reproduced.** For across-column pairs whose barrels share an arc, EN lowers signal
(0.478 to 0.416) and noise (0.337 to 0.262) correlations (p = 0.0001). For pairs whose barrels share
a row, noise correlation is lower in EN (p = 0.0001) but signal correlation is not (0.446 against
0.439, p = 0.38; stored p-values p = 0.081).

### fig6-l4-within-column-correlations

> In L4, EN increased both signal (0.681 to 0.774) and noise (0.241 to 0.356) correlations between
> co-columnar neurons.

In [ ]:
rng = np.random.default_rng(605)
EXP6 = {("NH", "signal"): (0.681, 0.011), ("EN", "signal"): (0.774, 0.005), ("NH", "noise"): (0.241, 0.006), ("EN", "noise"): (0.356, 0.002)}
for resp in ("methods", "stored"):
    for rr, barrels in (("A: any barrel", None), ("B: C1-C3", ["c1", "c2", "c3"])):
        pp = PAIRS[resp] if barrels is None else ssc10.pair_table([f for f in F if f["layer"] == "L4"], resp, barrels)
        p = pp[(pp.layer == "L4") & pp.same]
        print(f"Responsiveness {resp}, reading {rr}: pairs NH {(p.group == 'NH').sum()}, EN {(p.group == 'EN').sum()}")
        for key in ("signal", "noise"):
            d, pv = ssc10.perm_test_means(p[p.group == "EN"][key], p[p.group == "NH"][key], rng)
            nh, en = p[p.group == "NH"][key], p[p.group == "EN"][key]
            print(f"  {key}: NH {nh.mean():.4f} +/- {ssc10.sem(nh):.4f}, EN {en.mean():.4f} +/- {ssc10.sem(en):.4f}, EN - NH {d:+.4f}, p = {pv:.4g};"
                  f" within tolerance NH {abs(nh.mean() - EXP6[('NH', key)][0]) <= EXP6[('NH', key)][1]}, EN {abs(en.mean() - EXP6[('EN', key)][0]) <= EXP6[('EN', key)][1]}")

**Result: partially reproduced.** In L4, EN raises within-column signal and noise correlations under
both readings and both responsiveness tests (p = 0.0001). With C1-C3 and the Methods' test signal
correlation goes from 0.596 to 0.777 and noise from 0.218 to 0.338 (paper 0.681 to 0.774, 0.241 to
0.356); only the EN signal value is within tolerance.

### figs5-2-neuropil-robustness

> The enrichment effects on tuning heterogeneity and point representation were qualitatively
> unchanged with neuropil subtraction at r = 0, 0.3 and 0.7 in L2/3 and L4; effects on signal
> correlations held at r = 0 and 0.3 but were lost at 0.7, and effects on noise correlations held at
> all levels.

dF/F is rebuilt from `rawTimeSeries` and `rawTimeSeriesNP_corr` at each weight (helper,
`_dff`), and the whole pipeline is re-run: responsiveness by the Methods' test, eBWs (E1, E2), the
z-scored responses, and the pair correlations. The correlation test is the one the Figure 5 S2
legend names, an ANOVA with multiple comparisons, read as a one-way ANOVA over NH and EN within and
across columns with Tukey comparisons; the permutation test is the variant.

In [ ]:
rng = np.random.default_rng(701)
NPRES = {}
t0 = time.time()
for rw in ssc10.NP_WEIGHTS:
    for f in F:
        P = np.full((f["n_rois"], 9), np.nan)
        for j, w in enumerate(ssc10.WHISKERS):
            e, b = f["np"][rw]["ev"][w], f["np"][rw]["bl"]
            if e.shape[1]:
                ok = np.isfinite(e).all(1) & np.isfinite(b).all(1)
                pj = np.ones(f["n_rois"])
                pj[ok] = ssc10.perm_diff_many(e[ok], b[ok], rng)[1]
                P[:, j] = pj
        f.setdefault("resp_np", {})[rw] = ssc10.bh(np.where(np.isnan(P), 1, P)).any(1)
    tb, ar = ssc10.roi_table(F, "methods", rng, ebw=True, ebw_med=False, np_weight=rw)
    tb["inb"] = tb.barrel != ""; tb["c123"] = tb.barrel.isin(["c1", "c2", "c3"])
    pp = ssc10.pair_table(F, "methods", np_weight=rw)
    NPRES[rw] = (tb, ar, pp)
    print(f"r = {rw}: responsive ROIs", tb.groupby(["layer", "group"]).size().to_dict(), f"({time.time() - t0:.0f} s)")

In [ ]:
rows = []
rng = np.random.default_rng(702)
for rw, (tb, ar, pp) in NPRES.items():
    rec = {"r": rw}
    for lay, sel in (("L23", None), ("L4", tb.c123.values)):
        e = tuning_entries(tb, ar, lay, sel)
        e = e[e.dist < 75]
        for ek in ("e1", "e2"):
            fn, fe = e[e.group == "NH"][ek].mean(), e[e.group == "EN"][ek].mean()
            rec[f"{lay} CW-tuned 0-75 {ek} NH/EN"] = f"{100 * fn:.0f}/{100 * fe:.0f}"
            rec[f"{lay} {ek} EN higher"] = fe > fn
        pr = pointrep(tb, ar, lay, "z", sel, rng, bins=np.array([0, 75]))
        rec[f"{lay} z 0-75 NH/EN"] = f"{pr.iloc[0]['NH']:.2f}/{pr.iloc[0]['EN']:.2f} (p={pr.iloc[0]['p']:.3g})"
        rec[f"{lay} z 0-75 EN {'higher' if lay == 'L23' else 'lower'}"] = (pr.iloc[0]["diff"] > 0) if lay == "L23" else (pr.iloc[0]["diff"] < 0)
    p = pp[pp.layer == "L23"].copy()
    p["cond"] = p.group + np.where(p.same, " in", " x")
    for key in ("signal", "noise"):
        tk = pairwise_tukeyhsd(p[key], p.cond)
        tt = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
        for a, b, name in (("EN in", "NH in", "in"), ("EN x", "NH x", "x")):
            r = tt[((tt.group1 == a) & (tt.group2 == b)) | ((tt.group1 == b) & (tt.group2 == a))].iloc[0]
            m_en, m_nh = p[p.cond == a][key].mean(), p[p.cond == b][key].mean()
            pperm = ssc10.perm_test_means(p[p.cond == a][key], p[p.cond == b][key], rng)[1]
            rec[f"{key} {name} NH/EN"] = f"{m_nh:.3f}/{m_en:.3f}"
            rec[f"{key} {name} Tukey p"] = float(r["p-adj"])
            rec[f"{key} {name} perm p"] = pperm
            rec[f"{key} {name} EN lower sig (Tukey)"] = (m_en < m_nh) and float(r["p-adj"]) < 0.05
    rows.append(rec)
npt = pd.DataFrame(rows).set_index("r").T
print(npt.to_string())

**Result: not reproduced.** At r = 0, 0.3 and 0.7 the L2/3 fraction CW-tuned at 0-75 um is lower in
EN, as without subtraction, and the L4 0-75 um response is lower in EN only at r = 0.7; the L2/3
response is higher in EN at every weight and the L4 fraction CW-tuned is higher in EN at every
weight. The across-column signal correlation reduction stays significant at r = 0.7 (Tukey
p = 0.036), where the paper finds it lost, and the within-column noise correlation reduction is not
significant at r = 0.7 (p = 0.80).

## Decoding (Figure 7)

Each ROI's classifier is a logistic regression of stimulus (one whisker) against blank on the
single-trial evoked response, fit on the training trials of a split and applied to the test trials;
the predicted class is the more probable one (Methods, Neural decoder). L2/3 responsive ROIs.

In [ ]:
def nearest_col(f, r):
    d = np.linalg.norm(f["pos"][r], axis=-1)
    d = np.where([f["ev"][w].shape[1] > 0 for w in ssc10.WHISKERS], d, np.inf)
    return int(np.nanargmin(d))


rng = np.random.default_rng(801)
rows = []
t0 = time.time()
for f in F:
    if f["layer"] != "L23":
        continue
    rows_r = np.where(f["resp_methods"])[0]
    for scheme in ("random", "kfold"):
        for balance in (False, True):
            splits = {}
            for r in rows_r:
                targets = {"nearest": nearest_col(f, r)}
                if f["barrel"][r]:
                    targets["CW"] = ssc10.WHISKERS.index(f["barrel"][r])
                for name, j in targets.items():
                    w = ssc10.WHISKERS[j]
                    x = np.concatenate([f["ev"][w][r], f["bl"][r]])
                    y = np.concatenate([np.ones(f["ev"][w].shape[1]), np.zeros(f["bl"].shape[1])])
                    if j not in splits:
                        splits[j] = ssc10.make_splits(f["ev"][w].shape[1], f["bl"].shape[1], rng, scheme, 500, balance)
                    tr, te = splits[j]
                    _, acc = ssc10.roi_probs(x, y, tr, te)
                    rows.append(dict(field=f["field"], mouse=f["mouse"], group=f["group"], roi=r, target=name, scheme=scheme,
                                     balance=balance, acc=acc.mean(), chance=max(y.mean(), 1 - y.mean()) if not balance else 0.5))
DEC1 = pd.DataFrame(rows)
print(f"single-ROI decoders: {len(DEC1)} fits of 400-500 splits each, {time.time() - t0:.0f} s")

### fig7b-single-roi-decoder

> Single-ROI decoders detected CW deflection against blank above chance, slightly better for EN
> cells than NH cells (fraction correct 0.65 against 0.63).

Readings: (i) splits, (a) 500 random 80/20 or (b) 5-fold repeated 100 times; (ii) balance, (a) all
trials or (b) blanks subsampled; (iii) ROIs, (a) over a barrel, tested on the CW, or (b) all
responsive ROIs, tested on the nearest column's whisker. The majority-class rate is printed as the
chance level for unbalanced readings (0.5 when balanced).

In [ ]:
rng = np.random.default_rng(802)
rows = []
for (target, scheme, balance), d in DEC1.groupby(["target", "scheme", "balance"]):
    nh, en = d[d.group == "NH"].acc, d[d.group == "EN"].acc
    diff, p = ssc10.perm_test_means(en, nh, rng)
    rows.append(dict(rois="(a) CW, over a barrel" if target == "CW" else "(b) nearest column, all", splits=scheme, balanced=balance,
                     NH=f"{nh.mean():.4f} +/- {ssc10.sem(nh):.4f}", EN=f"{en.mean():.4f} +/- {ssc10.sem(en):.4f}", p=p,
                     n=(len(nh), len(en)), chance_NH=round(d[d.group == "NH"].chance.mean(), 3), chance_EN=round(d[d.group == "EN"].chance.mean(), 3),
                     within=int(abs(nh.mean() - 0.63) <= 0.005) + int(abs(en.mean() - 0.65) <= 0.004)))
print(pd.DataFrame(rows).to_string(index=False))
print("\nBy mouse, reading (a)(a)(a):")
d = DEC1[(DEC1.target == "CW") & (DEC1.scheme == "random") & (~DEC1.balance)]
print(d.groupby(["group", "mouse"]).acc.agg(["mean", "size"]).round(4).to_string())

**Result: partially reproduced.** Single-ROI decoders are above chance in both groups and better in
EN under every combination of readings (for CW decoders over a barrel, 500 random splits, all
trials: NH 0.615, EN 0.634, p = 0.0004; majority-class rate 0.52). The values are 0.015 below the
paper's in both groups, outside the tolerances of 0.005 and 0.004, in every combination.

### fig7c-population-decoder

> Detection by population decoders of randomly chosen, simultaneously imaged ROIs improved with
> ensemble size and was better for EN than NH for ensembles of 3 or more neurons.

Ensembles of 1, 2, 3, 5, 10, 15, 20, 25 and 30 responsive ROIs of a field, the lesser of 500 and
N-choose-K per field and size (Methods). Classifiers are fit on 50 random 80/20 splits per field and
whisker, shared by all ROIs of the field (the Methods give no number of repetitions for ensembles;
50 is a standard choice made for time). The ensemble sums each class's probability over its ROIs
(normalisation (a)), or sums each ROI's probability z-scored over the test trials (b).

In [ ]:
N_SPLIT_ENS = 50
SIZES = [1, 2, 3, 5, 10, 15, 20, 25, 30]


def field_probs(f, rows_r, rng, n_split=N_SPLIT_ENS):
    """For each whisker: (P: splits x ROIs x trials, y, test masks)."""
    out = {}
    for j, w in enumerate(ssc10.WHISKERS):
        ns = f["ev"][w].shape[1]
        if ns == 0:
            continue
        nb = f["bl"].shape[1]
        tr, te = ssc10.make_splits(ns, nb, rng, "random", n_split, False)
        y = np.concatenate([np.ones(ns), np.zeros(nb)])
        P = np.stack([ssc10.roi_probs(np.concatenate([f["ev"][w][r], f["bl"][r]]), y, tr, te)[0] for r in rows_r], axis=1)
        # normalisation (b): each ROI's probability z-scored over the test trials of the split
        Pm = np.where(te[:, None, :], P, np.nan)
        Z = (P - np.nanmean(Pm, 2, keepdims=True)) / (np.nanstd(Pm, 2, keepdims=True) + 1e-12)
        out[j] = (P, Z, y, te)
    return out


def ens_acc(P, Z, y, te, members, norm):
    if norm == "a":
        s = P[:, members, :].sum(1)
        score = s - (len(members) - s)                          # summed P(stim) minus summed P(blank)
    else:
        score = Z[:, members, :].sum(1)
    pred = score > 0
    return (((pred == (y > 0.5)) & te).sum(1) / te.sum(1)).mean()


rng = np.random.default_rng(803)
rows = []
t0 = time.time()
PROBS = {}
for fi, f in enumerate(F):
    if f["layer"] != "L23":
        continue
    rows_r = np.where(f["resp_methods"])[0]
    N = len(rows_r)
    probs = field_probs(f, rows_r, rng)
    PROBS[fi] = (rows_r, probs)
    col = np.array([ssc10.WHISKERS.index(f["barrel"][r]) if f["barrel"][r] else nearest_col(f, r) for r in rows_r])
    for K in SIZES:
        if K > N:
            continue
        n_ens = min(500, comb(N, K))
        seen = set()
        while len(seen) < n_ens:
            seen.add(tuple(sorted(rng.choice(N, K, replace=False))))
        for ens in seen:
            ens = np.array(ens)
            cen = f["pos"][rows_r[ens]].mean(0)                  # 9 x 2, centroid relative to each barrel
            dcen = np.linalg.norm(cen, axis=-1)
            dcen = np.where([j in probs for j in range(9)], dcen, np.inf)
            j_close = int(np.argmin(dcen))
            rec = dict(fi=fi, field=f["field"], group=f["group"], K=K)
            for norm in ("a", "b"):
                rec[f"close_{norm}"] = ens_acc(*probs[j_close], ens, norm)
                cws = sorted(set(col[ens]) & set(probs))
                rec[f"cws_{norm}"] = np.mean([ens_acc(*probs[j], ens, norm) for j in cws])
            rows.append(rec)
ENS = pd.DataFrame(rows)
print(f"random ensembles: {len(ENS)} in {time.time() - t0:.0f} s; per field and size:")
print(ENS.groupby(["group", "K"]).size().unstack().to_string())

In [ ]:
rng = np.random.default_rng(804)
res = []
for tw in ("close", "cws"):
    for norm in ("a", "b"):
        key = f"{tw}_{norm}"
        rec = dict(tested_whisker="(a) closest column" if tw == "close" else "(b) CWs of members", normalisation=norm)
        p_pool, p_field, means = [], [], []
        for K in SIZES:
            d = ENS[ENS.K == K]
            nh, en = d[d.group == "NH"][key], d[d.group == "EN"][key]
            _, pp = ssc10.perm_test_means(en, nh, rng)
            fm = d.groupby(["group", "field"])[key].mean()
            _, pf = ssc10.perm_test_means(fm["EN"], fm["NH"], rng)
            rec[f"K={K} NH/EN"] = f"{nh.mean():.3f}/{en.mean():.3f}"
            rec[f"K={K} p pooled/fields"] = f"{pp:.2g}/{pf:.2g}"
            if K >= 3:
                p_pool.append((en.mean() > nh.mean()) and pp < 0.001)
                p_field.append((fm["EN"].mean() > fm["NH"].mean()) and pf < 0.001)
            means.append((nh.mean(), en.mean()))
        rec["rises with size (NH, EN)"] = (bool(np.all(np.diff([m[0] for m in means]) > 0)), bool(np.all(np.diff([m[1] for m in means]) > 0)))
        rec["K>=3 EN>NH p<0.001 pooled"] = f"{sum(p_pool)}/{len(p_pool)}"
        rec["K>=3 EN>NH p<0.001 per field"] = f"{sum(p_field)}/{len(p_field)}"
        res.append(rec)
print(pd.DataFrame(res).T.to_string())
print("\nFields per group:", ENS.groupby("group").field.nunique().to_dict())

**Result: reproduced, under one reading.** With ensembles pooled as units, performance rises with
ensemble size in both groups (closest-column whisker, normalisation a: NH 0.612 to 0.676, EN 0.628
to 0.711 from 1 to 30 ROIs) and EN exceeds NH with p < 0.001 at every size from 3 on, under all
four combinations of tested whisker and normalisation. With per-field means as units no size reaches
p < 0.001 (p from 0.025 to 0.38 over 11 fields per group).

### fig7d-clustered-ensemble-topography

> Spatially clustered ensembles detected the reference whisker best near its column centre; EN
> decoders outperformed NH decoders for ensembles centred 0-77 and 77-154 um from the column centre,
> and EN performed worse than NH farther away (e.g. 308-385 um).

For each field and reference whisker, k-means on the responsive ROIs' positions normalised to the
reference barrel (`ROI_pos_barrel_norm`), for every k from 2 to the number of responsive ROIs
(Methods); clusters of 2 to 42 ROIs are ensembles. k-means is `scipy.cluster.vq.kmeans2` with
k-means++ starts, one run per k (a standard choice). Each ensemble detects its reference whisker
against blank (normalisation (a), the 50 splits above); its centroid's distance to the reference
column centre is in um (`ROI_pos_barrel`), in 77 um bins.

In [ ]:
from scipy.cluster.vq import kmeans2

rng = np.random.default_rng(805)
rows = []
t0 = time.time()
for fi, (rows_r, probs) in PROBS.items():
    f = F[fi]
    N = len(rows_r)
    for j in probs:
        X = f["posn"][rows_r, j]
        ok = np.isfinite(X).all(1)
        for k in range(2, N + 1):
            _, lab = kmeans2(X[ok], k, minit="++", seed=rng)
            for c in range(k):
                ens = np.where(ok)[0][lab == c]
                if not 2 <= len(ens) <= 42:
                    continue
                d = np.linalg.norm(f["pos"][rows_r[ens], j].mean(0))
                rows.append(dict(fi=fi, field=f["field"], group=f["group"], ref=j, k=k, K=len(ens), dist=d,
                                 acc=ens_acc(*probs[j], ens, "a")))
CLU = pd.DataFrame(rows)
CLU["bin"] = (CLU.dist // 77).astype(int)
print(f"clustered ensembles: {len(CLU)} in {time.time() - t0:.0f} s; per field: ",
      CLU.groupby(["group", "field"]).size().groupby("group").agg(["mean", "std"]).round(0).to_dict("index"))

In [ ]:
lab = {b: f"{77 * b}-{77 * (b + 1)}" for b in range(10)}
c = CLU[CLU.bin < 8]
summ = c.groupby(["bin", "group"]).acc.agg(["mean", "size"]).unstack("group")
summ.index = [lab[b] for b in summ.index]
print(summ.round(4).to_string())
pooled = c.groupby("bin").acc.mean()
print("Both groups pooled, mean per bin:", {lab[b]: round(v, 4) for b, v in pooled.items()})
print("\nBin with the highest mean performance: pooled", lab[pooled.idxmax()], "; NH", lab[c[c.group == "NH"].groupby("bin").acc.mean().idxmax()],
      ", EN", lab[c[c.group == "EN"].groupby("bin").acc.mean().idxmax()])
# Reading (a): per distance bin, two-way ANOVA with factors enrichment and ensemble size
print("\nReading (a): two-way ANOVA (enrichment x ensemble size) within each bin")
for b in (0, 1, 4):
    d = c[c.bin == b]
    an = anova_lm(smf.ols("acc ~ C(group) + C(K)", data=d).fit(), typ=2)
    print(f"  {lab[b]}: EN - NH {d[d.group == 'EN'].acc.mean() - d[d.group == 'NH'].acc.mean():+.4f},"
          f" enrichment F = {an.loc['C(group)', 'F']:.1f}, p = {an.loc['C(group)', 'PR(>F)']:.3g}")
# Reading (b): two-way ANOVA with factors enrichment and distance bin, then per-bin comparisons
an = anova_lm(smf.ols("acc ~ C(group) * C(bin)", data=c).fit(), typ=2)
print("\nReading (b): two-way ANOVA (enrichment x distance bin)")
print(an.round(4).to_string())
c2 = c.assign(cond=c.group + "_" + c.bin.astype(str))
tk = pairwise_tukeyhsd(c2.acc, c2.cond)
tt = pd.DataFrame(tk._results_table.data[1:], columns=tk._results_table.data[0])
for b in (0, 1, 4):
    r = tt[(tt.group1 == f"EN_{b}") & (tt.group2 == f"NH_{b}")]
    print(f"  {lab[b]}: Tukey EN vs NH, meandiff (NH - EN) {float(r['meandiff'].iloc[0]):+.4f}, p-adj {float(r['p-adj'].iloc[0]):.3g}")
fig, ax = plt.subplots(figsize=(6, 3.5))
for g, col in (("NH", "k"), ("EN", "tab:red")):
    m = c[c.group == g].groupby("bin").acc.mean()
    ax.plot([77 * b + 38 for b in m.index], m.values, "o-", color=col, label=g)
ax.set_xlabel("ensemble centroid distance from reference column centre (um)"); ax.set_ylabel("fraction correct"); ax.legend()
plt.tight_layout(); plt.show()
print("Plotted means range from", round(c.groupby(["group", "bin"]).acc.mean().min(), 3), "to", round(c.groupby(["group", "bin"]).acc.mean().max(), 3))

**Result: reproduced.** Clustered ensembles (NH 4235, EN 11038 per field, paper 4242 and 9630)
perform best at 0-77 um with both groups pooled (0.679, against 0.664 at 77-154 um) and in EN
(0.686), and fall off with distance; in NH the 0-77 and 77-154 um bins are close (0.640 and 0.644). EN beats
NH at 0-77 and 77-154 um and NH beats EN at 308-385 um under both readings of the ANOVA (reading a,
enrichment p = 4e-37, 2e-67 and 8e-22; reading b, Tukey p < 0.001).

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| table1-mice-and-fields | reproduced |
| l23-cell-counts | reproduced |
| l4-cell-counts | not reproduced |
| fig1f-gcamp-matches-spikes | blocked: data missing |
| fig2a-l23-point-representation-stronger | reproduced, with the stored responsiveness and the blank-median z |
| fig2a-l23-flanks-suppressed | partially reproduced |
| fig2b-2d-point-representation | reproduced |
| fig2c-l4-point-representation-weaker | not reproduced |
| fig2c-l4-sharper-than-l23 | reproduced |
| figs2-1-column-frame | not reproduced |
| figs2-2-barrel-size-unchanged | partially reproduced |
| figs3-1-l23-salt-and-pepper | partially reproduced |
| discussion-l23-nh-in-column-tuning | partially reproduced |
| figs3-1-l4-nh-tuning | not reproduced |
| fig3a-l23-enrichment-more-cw-tuned | not reproduced |
| fig3c-l4-enrichment-more-cw-tuned | reproduced |
| fig4a-l23-ranked-rf-sharper | reproduced |
| fig4b-l23-cw-and-best-sw-stronger | reproduced |
| fig4c-spontaneous-activity | reproduced, with spontaneous activity without baseline subtraction |
| figs4-1-absolute-dff | not reproduced |
| figs4-2-barrel-vs-septal | reproduced |
| fig4d-l23-cw-stronger | reproduced, with the stored responsiveness |
| fig4d-l23-across-row-sharpening | partially reproduced |
| fig4d-l23-within-row-broadening | partially reproduced |
| fig4ae-l4-weaker-responses | not reproduced |
| fig5a-l23-signal-correlation-boundary | partially reproduced |
| fig5b-l23-noise-correlation-boundary | partially reproduced |
| fig5eh-l23-correlation-gradient-at-edge | reproduced, with the stored responsiveness |
| figs5-1-row-and-arc-boundaries | not reproduced |
| fig6-l4-within-column-correlations | partially reproduced |
| figs5-2-neuropil-robustness | not reproduced |
| fig7b-single-roi-decoder | partially reproduced |
| fig7c-population-decoder | reproduced, with ensembles as units |
| fig7d-clustered-ensemble-topography | reproduced |

How the analysis differs from the paper's. The responsiveness test is the Methods' one-sided test of
means; the p-values stored in the data set come from a two-sided test on medians (`permTestType`
'median'), and the two agree on 85% of ROIs. Both are shown, and four outcomes rest on the stored
ones. The equivalent best whiskers are computed by three readings the claims file names, since
neither the Methods nor the code say how; all three put many more L2/3 NH cells near a column centre
in the tuned group than the paper reports, while L4 values match. Barrel radii come from the
normalised ROI positions, not from barrel areas. Ensemble decoders use 50 splits per field and
whisker, and one k-means run per k.

What was found about the data set. The counts of mice and fields match Table 1, and the NH L2/3
cell count is exact; the EN L2/3 and the L4 totals differ from the paper's (1147 against 1257, 362
against 322, 615 against 586), and `processed_data.mat` holds more ROIs than `reduced_data.mat`
with no stated rule. One NH field has no E-row whisker trials. The stored dF/F follows the
pipeline's 5-frame filter and 5th-percentile F0, not the Methods' 4 frames and 20th percentile. The
label code of `Stimuli` and the blank-trial selection of `sponTrace` are not documented. No barrel
areas are stored, and the cell-attached recordings of Figure 1F are not in the data set.